# V3 Pilot 2080

The masked concept pilot of Section 8. The subset supervised arm W computes its subset attention through `subset_attention_levels_exact_bwd`, the cache the Section 7 benchmarks measure. W0 has no subset term and does not use the cache. The cache provides one additional output for the pilot, the log normalizer of each subset, with its exact gradient in the same backward pass, tested in cell 1.

**Machine.** RTX 2080 SUPER, 8 GiB, Windows, Python 3.12, torch 2.5.1 with CUDA 12.1. One seed, 367. Times reported here are for this machine only.

**Arms.** Three training runs, each two epochs on the same microbatch plans.

| Arm | Context | Subset term | Update rule |
|---|---|---|---|
| W0, registered | every other feature of the record | none | one update per microbatch |
| W0, accumulated | every other feature of the record | none | accumulated, one update per 32 or more targets |
| W | every other feature of the record | run sampler subsets, Horvitz Thompson weighted | accumulated |

**Data.** In `E:\apriori-ml-v3\pilot`. The cleaned pilot data (`mc3_pilot_data_d5.pkl.lz4`), the ranked contexts (`mc3_contexts_d5.pkl.lz4`), the decile edges (`mc3_decile_edges.pkl.lz4`) and the log of removed values (`mc3_pilot_data_d5_removed.pkl.lz4`), each hashed and checked by fingerprint before use. The ranked contexts file is read by no arm in this notebook. It is loaded by cell 2 and hashed into every contract, so it ships with the release as a contract input. No file outside the repository is read.

**Library.** `E:\apriori-ml-v3\src`. Six files pinned by hash in cell 2, three recorded from disk and pinned at the release freeze. The nine hashes enter every contract and every saved record. The pilot's subset families are built per target without that target, and their tries are built from the subset list by the canonical builder. The implemented run trie generator of Section 7 is not used for these target excluded families.

**Order of cells.** Nothing trains until cells 1 to 5 pass. Cells 7 to 9 read saved files and train nothing.

| Cell | Purpose |
|---|---|
| 1 | Tests of the cache's log normalizer output against an independent float64 reference |
| 2 | Configuration, library identity and data |
| 3 | Agreement gate: the cache against the reference traversal, pointer jumping, for W |
| 4 | Gate record |
| 5 | Training harness for the three arms, epoch plans and held out set checks |
| 6 | Training: the three arms, about 5.6 hours |
| 7 | Paired intervals and bin results (the report) |
| 8 | Matched cost study and agreement checks: cache, reference traversal and direct computation |
| 9 | Tables for Section 8 and the unrounded hours and ratios for Section 8.6 ||
| 10 | Tables for Section 9 |

## Cell 1. The cache's log normalizer output

The pilot's prediction head reads each subset's output vector and its log normalizer per head, L = M + log S, where M is the largest score in the subset and S the sum of exp(score − M). `subset_attention_levels_exact_bwd(z, V, plan, return_lse=True)` now returns both. Without `return_lse` it returns the output alone, as before.

**Gradient.** With weight w = exp(z_i − M_P) / S_P for fact i in subset P, an incoming gradient l_P on L_P adds w × l_P to the score gradient of fact i and nothing to its value gradient. The backward folds this into the term it already carries for each subset.

**Test.** One forest of three tries over separate row blocks (10, 12 and 9 rows; 300 sampled, 400 sampled and all 511 subsets), 4 channels, 64 values per channel, scores scaled by 3, TF32 off. The cache runs in float32. The reference computes every subset's softmax and log sum exp on its own, in float64, with autograd. Fixed random upstream gradients drive four losses. Every tensor is checked finite. The cell raises an error if any check fails, so a passing run is the gate.

Printed labels, in order:

- `set_attention_model.py sha256`: hash of the library file under test, for the record.
- `forest`: number of tries, rows `N`, subsets `P`, channels `C`, values per channel `D`.
- The consistency table, cache only, columns `check`, `value`, `tol`, `pass`:
  - `two outputs with pooled loss vs pooled-only call` for the output, `dz` and `dV`. The new call with a loss on the output alone, against the original call. Output by the largest absolute difference rule, gradients by relative Euclidean error.
  - `combined dz vs (pooled loss dz + L only dz)`. The combined loss's score gradient against the sum of the two separately tested contributions.
  - `combined dV vs pooled loss dV`. L adds nothing to value gradients, so these agree.
- `ALL PASS`: True when every case and every consistency check passes. Otherwise the cell raises.
- The results table, one row per case:
  - `case`. `pooled-only call` is the original signature. `two outputs, pooled loss` returns L but uses only the output. `L only` uses only L. `combined` uses both.
  - `out_err`, `out_tol`. Largest absolute output difference from the reference, and its tolerance 1e-6 + 1e-4 × largest reference value.
  - `L_err`, `L_tol`. The same for L, on cases that return it.
  - `dz_err`, `dz_rule`. Score gradient error. Relative Euclidean error at most 1e-3, or absolute Euclidean error at most 1e-6 when the reference gradient is zero.
  - `dV_err`, `dV_rule`. The same for the value gradient.
  - `dV_exactly_zero`. `L only` case. The cache's value gradient must be exactly zero.
  - `finite`. Every reference and cache tensor of the row is finite.
  - `pass`. Every check of the row met.

In [2]:
# Estimated run time: under 1 minute on the 2080 (Triton compiles the new kernel specialization on first use). Fresh kernel.
# Focused tests of the optional L output of subset_attention_levels_exact_bwd. Fused path in float32 against an independent
# per subset softmax and log sum exp reference in float64 with autograd, on a three tree forest with row offsets.
# Raises if any check fails.
import os, sys, hashlib, itertools
import numpy as np
import pandas as pd
import torch

SRC = r"E:\apriori-ml-v3\src"
SEED, C, D = 367, 4, 64
BLOCK_N = (10, 12, 9)
SUBSETS_PER_BLOCK = (300, 400, 511)              # the third block is exhaustive at n = 9 (2^9 - 1 = 511 subsets)
SCORE_SCALE = 3.0                                # spreads scores so the running maximum changes along paths
TOL_OUT, TOL_GRAD, TOL_ZERO = (1e-6, 1e-4), 1e-3, 1e-6   # the suite's float32 rules

sys.path.insert(0, SRC)
import set_attention_model as sam
import set_attention_cache, set_attention_data
from set_attention_cache import build_prefix_trie_canonical
from set_attention_data import forest_level_plan_np, plan_to_device
for mod in (sam, set_attention_cache, set_attention_data):
    assert os.path.normcase(os.path.abspath(mod.__file__)).startswith(os.path.normcase(SRC) + os.sep), \
        f"{mod.__name__} loaded from {mod.__file__}, restart the kernel"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

print("set_attention_model.py sha256", sha256_file(os.path.join(SRC, "set_attention_model.py")))
dev = torch.device("cuda")
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# ----- forest: three tries over separate row blocks, canonical order inside each -----
def block_subsets(n, k):
    if k >= 2 ** n - 1:
        return sorted(S for r in range(1, n + 1) for S in itertools.combinations(range(n), r))
    seen = set()
    while len(seen) < k:
        r = int(rng.integers(1, n + 1))
        seen.add(tuple(sorted(rng.choice(n, r, replace=False).tolist())))
    return sorted(seen)

offsets = np.concatenate([[0], np.cumsum(BLOCK_N)[:-1]]).astype(np.int64)
subs_b = [block_subsets(n, k) for n, k in zip(BLOCK_N, SUBSETS_PER_BLOCK)]
tries = [build_prefix_trie_canonical(s) for s in subs_b]
plan = plan_to_device(forest_level_plan_np(tries, offsets), dev)
subs_global = [tuple(int(off) + i for i in S) for off, s in zip(offsets, subs_b) for S in s]   # forest order
N, P = int(sum(BLOCK_N)), len(subs_global)
assert plan["inv"].numel() == P, "plan subset count differs from the forest's subsets"
print(f"forest: {len(tries)} tries, N {N} rows, P {P} subsets, C {C}, D {D}")

z64 = torch.randn(N, C, dtype=torch.float64, device=dev) * SCORE_SCALE
V64 = torch.randn(N, C, D, dtype=torch.float64, device=dev)
gO64 = torch.randn(P, C, D, dtype=torch.float64, device=dev)
gL64 = torch.randn(P, C, dtype=torch.float64, device=dev)

# ----- independent reference: per subset softmax and log sum exp, float64, autograd -----
def ref_forward(z, V):
    outs, Ls = [], []
    for S in subs_global:
        idx = torch.as_tensor(S, device=dev)
        zs = z[idx]                                          # [k, C]
        L = torch.logsumexp(zs, 0)                           # [C]
        w = torch.exp(zs - L)                                # [k, C]
        outs.append(torch.einsum("kc,kcd->cd", w, V[idx]))
        Ls.append(L)
    return torch.stack(outs), torch.stack(Ls)

def loss_of(kind, out, L, gO, gL):
    if kind in ("pooled-only call", "two outputs, pooled loss"):
        return (out * gO).sum()
    if kind == "L only":
        return (L * gL).sum()
    return (out * gO).sum() + (L * gL).sum()

def ref_case(kind):
    z = z64.clone().requires_grad_(); V = V64.clone().requires_grad_()
    out, L = ref_forward(z, V)
    dz, dV = torch.autograd.grad(loss_of(kind, out, L, gO64, gL64), (z, V), allow_unused=True)
    dz = torch.zeros_like(z) if dz is None else dz
    dV = torch.zeros_like(V) if dV is None else dV                           # L only: the loss does not reach V
    return out.detach(), L.detach(), dz.detach(), dV.detach()

def fused_case(kind):
    z = z64.float().requires_grad_(); V = V64.float().requires_grad_()
    if kind == "pooled-only call":
        out, L = sam.subset_attention_levels_exact_bwd(z, V, plan), None
    else:
        out, L = sam.subset_attention_levels_exact_bwd(z, V, plan, return_lse=True)
    loss_of(kind, out, L, gO64.float(), gL64.float()).backward()
    torch.cuda.synchronize()
    dz = torch.zeros_like(z) if z.grad is None else z.grad
    dV = torch.zeros_like(V) if V.grad is None else V.grad
    return out.detach().double(), None if L is None else L.detach().double(), dz.double(), dV.double()

def max_abs_rule(a, b):
    err, tol = float((a - b).abs().max()), TOL_OUT[0] + TOL_OUT[1] * float(b.abs().max())
    return err, tol, err <= tol

def grad_rule(a, b):
    """Relative Euclidean error <= 1e-3, or absolute Euclidean error <= 1e-6 against a zero reference."""
    nb, nd = float(b.norm()), float((a - b).norm())
    if nb > 0:
        return nd / nb, "relative <= 1e-3", nd / nb <= TOL_GRAD
    return nd, "absolute <= 1e-6", nd <= TOL_ZERO

def finite(*ts):
    return all(bool(torch.isfinite(t).all()) for t in ts if t is not None)

KINDS = ["pooled-only call", "two outputs, pooled loss", "L only", "combined"]
fused = {k: fused_case(k) for k in KINDS}
rows = []
for k in KINDS:
    o_r, L_r, dz_r, dV_r = ref_case(k)
    o_f, L_f, dz_f, dV_f = fused[k]
    r = {"case": k, "finite": finite(o_r, L_r, dz_r, dV_r, o_f, L_f, dz_f, dV_f)}
    ok = r["finite"]
    r["out_err"], r["out_tol"], ok_o = max_abs_rule(o_f, o_r); ok &= ok_o
    if L_f is not None:
        r["L_err"], r["L_tol"], ok_l = max_abs_rule(L_f, L_r); ok &= ok_l
    r["dz_err"], r["dz_rule"], ok_z = grad_rule(dz_f, dz_r); ok &= ok_z
    r["dV_err"], r["dV_rule"], ok_v = grad_rule(dV_f, dV_r); ok &= ok_v
    if k == "L only":
        r["dV_exactly_zero"] = bool((dV_f == 0).all()); ok &= r["dV_exactly_zero"]
    r["pass"] = bool(ok)
    rows.append(r)
results = pd.DataFrame(rows)

# ----- consistency across cases, cache only -----
A, B, Lo, Cb = (fused[k] for k in KINDS)
crows = []
e, t, ok = max_abs_rule(B[0], A[0])
crows.append({"check": "two outputs with pooled loss vs pooled-only call, output max abs", "value": e, "tol": t, "pass": ok})
for name, a, b in (("dz", B[2], A[2]), ("dV", B[3], A[3])):
    v, rule, ok = grad_rule(a, b)
    crows.append({"check": f"two outputs with pooled loss vs pooled-only call, {name} ({rule})", "value": v, "tol": TOL_GRAD if "relative" in rule else TOL_ZERO, "pass": ok})
v, rule, ok = grad_rule(Cb[2], B[2] + Lo[2])
crows.append({"check": f"combined dz vs (pooled loss dz + L only dz) ({rule})", "value": v, "tol": TOL_GRAD if "relative" in rule else TOL_ZERO, "pass": ok})
v, rule, ok = grad_rule(Cb[3], B[3])
crows.append({"check": f"combined dV vs pooled loss dV ({rule})", "value": v, "tol": TOL_GRAD if "relative" in rule else TOL_ZERO, "pass": ok})
consistency = pd.DataFrame(crows)
print(consistency.to_string(index=False))

all_pass = bool(results["pass"].all()) and bool(consistency["pass"].all())
print("ALL PASS:", all_pass)
if not all_pass:
    print(results.to_string(index=False))
    raise AssertionError("log normalizer output tests failed; see the tables above")
results

set_attention_model.py sha256 d7a7e72a2a111bb40e45cbc61ed844349e51ed15c7c22749a817893eb8eee345
forest: 3 tries, N 31 rows, P 1211 subsets, C 4, D 64
                                                                  check        value      tol  pass
       two outputs with pooled loss vs pooled-only call, output max abs 0.000000e+00 0.000374  True
two outputs with pooled loss vs pooled-only call, dz (relative <= 1e-3) 1.898635e-07 0.001000  True
two outputs with pooled loss vs pooled-only call, dV (relative <= 1e-3) 1.291158e-07 0.001000  True
         combined dz vs (pooled loss dz + L only dz) (relative <= 1e-3) 2.409847e-07 0.001000  True
                       combined dV vs pooled loss dV (relative <= 1e-3) 1.248389e-07 0.001000  True
ALL PASS: True


,case,finite,out_err,out_tol,dz_err,dz_rule,dV_err,dV_rule,pass,L_err,L_tol,dV_exactly_zero
0,pooled-only call,True,6.321016e-07,0.000374,3.741347e-07,relative <= 1e-3,1.745108e-07,relative <= 1e-3,True,NaN,NaN,NaN
1,"two outputs, pooled loss",True,6.321016e-07,0.000374,3.481325e-07,relative <= 1e-3,1.598397e-07,relative <= 1e-3,True,5.887952e-07,0.000917,NaN
2,L only,True,6.321016e-07,0.000374,1.996133e-07,relative <= 1e-3,0.000000e+00,absolute <= 1e-6,True,5.887952e-07,0.000917,True
3,combined,True,6.321016e-07,0.000374,3.209153e-07,relative <= 1e-3,1.738337e-07,relative <= 1e-3,True,5.887952e-07,0.000917,NaN


## Cell 2. Configuration, library identity and data

Settings: budget M 2,048 with run length n for W, 32 targets per microbatch, interleaved column order with seed tag 131, AdamW at learning rate 3e-4 with weight decay 0.01, learning rate by target exposure (warmup over the first 10% of one epoch's targets, then cosine over 8 epochs' targets), one update per 32 or more accumulated targets, 2 epochs run, 16 loader workers, seed 367, 2,000 bootstrap resamples. Under accumulation, a microbatch is not an optimizer update.

The cell must start in a fresh kernel. It refuses to run if any library or pilot module is already imported, puts `E:\apriori-ml-v3\src` first on the import path, checks that every module loads from there, and hashes every library file. Six files are pinned: `set_attention.py`, `set_attention_data.py`, `set_attention_model.py` (d7a7e72a, the version with the log normalizer output tested in cell 1), `set_attention_bench.py`, `masked_concept_model.py` and `mc_batched.py`. Three are recorded from disk and pinned at the release freeze: `set_attention_cache.py`, which carries the run trie generator of Section 7, `mc_pilot_data.py` and `mc_walk_jump_custom.py`. All nine hashes enter every training contract and every saved record.

Data identity. The four data files in `E:\apriori-ml-v3\pilot` are hashed, and the fingerprint labels stored inside the pilot data and the contexts are compared with the recorded identities, as is the decile edges file hash. The removed value log is hashed and recorded. No file outside the repository is read.

Printed labels, in order:

- `library`: each library file, the first 16 characters of its sha256, and `ok` (True for a pinned file that matches its pin, True by definition for a recorded file).
- `environment`: GPU, torch, CUDA, Python, platform and the TF32 settings (both off).
- `data`: per file, `sha16` (first 16 of its sha256), `mib`, and where a recorded identity exists, `stored_label` and `label_ok`.
- `counts`: training filings, held out filings, training occurrences (1,234,902), held out targets (136,446) and microbatches per epoch (126,767), with `ok` when a recorded value exists and matches. `P` agree across the three layouts. and matches.

In [1]:
# Estimated run time: about 1 minute on the 2080 (hashes four data files, loads the pilot data and contexts, builds one
# epoch plan). Fresh kernel only.
# Cell 2. Configuration, library identity and data for the pilot.
import os, sys, time, math, hashlib, json, platform, datetime
import numpy as np
import pandas as pd

PILOT_MODULES = ("set_attention", "set_attention_cache", "set_attention_data", "set_attention_model", "set_attention_bench",
                 "masked_concept_model", "mc_batched", "mc_pilot_data", "mc_walk_jump_custom", "apriori")
already = sorted(m for m in sys.modules if m.split(".")[0] in PILOT_MODULES)
assert not already, f"library or pilot modules already imported, restart the kernel: {already}"
import torch

ROOT = r"E:\apriori-ml-v3"
SRC = os.path.join(ROOT, "src")
PILOT_DIR = os.path.join(ROOT, "pilot")
RUN_DIR = os.path.join(PILOT_DIR, "runs")                        # curves and checkpoints of this notebook
PILOT = os.path.join(PILOT_DIR, "mc3_pilot_data_d5.pkl.lz4")
CTXF = os.path.join(PILOT_DIR, "mc3_contexts_d5.pkl.lz4")
EDGES = os.path.join(PILOT_DIR, "mc3_decile_edges.pkl.lz4")
REMOVED = os.path.join(PILOT_DIR, "mc3_pilot_data_d5_removed.pkl.lz4")

# ----- settings -----
M_W, RUN_FACTOR, N_T, BATCH, ORDER_SEED_TAG = 2048, 1, 12, 32, 131
LR, WD, WARMUP_FRAC, MIN_TARGETS = 3e-4, 0.01, 0.10, 32
EPOCHS_RUN, EPOCHS_REGISTERED = 2, 8
WORKERS, SEED, EVAL_CHUNK, N_PROBE, LOADER_TIMEOUT, MAX_STALLS, BOOT = 16, 367, 1024, 20, 300, 20, 2000
MASK_RATE = 0.15                                                 # the dataset constructor's argument; column targets are explicit
BANDS = [(0, 1, "0 to 1"), (1, 3, "1 to 3"), (3, 10, "3 to 10"), (10, np.inf, "10+")]

# ----- recorded identities -----
FROZEN_FILE_SHA256 = {
    "set_attention.py":        "8580eb3ce583edec900c993d58793e98d2ef4e9c434f0dddc5a58991918f0d52",
    "set_attention_data.py":   "fd4c5a486477ccc30a7c202891ffbc579dc02b2bc5ae5988e94cbe6d7ac23cea",
    "set_attention_model.py":  "d7a7e72a2a111bb40e45cbc61ed844349e51ed15c7c22749a817893eb8eee345",
    "set_attention_bench.py":  "80b9a7eddb88ddee8948d997808643e476328e915b7ac09847e3ff25f64ea72a",
    "masked_concept_model.py": "f6b49be2e07791e1e8f39729f881e4488a85943f8de35a84ebf83071d751f330",
    "mc_batched.py":           "571185cc24b467ee1955561b19348522548e328ecd7b0aa1767b8e514d78c43e",
}
LIBRARY_FILES = list(FROZEN_FILE_SHA256) + ["set_attention_cache.py", "mc_pilot_data.py", "mc_walk_jump_custom.py"]   # recorded from disk, pinned at the release freeze
FACTS_FP, CONTEXTS_FP, EDGES_SHA16 = "1d6ee181e9fd1807", "bb82e1d833c2983b", "F6270354A9B66881"
RECORD = {"training occurrences": 1_234_902, "held out targets": 136_446, "microbatches per epoch": 126_767}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# ----- import path and library identity -----
sys.path = [p for p in sys.path if os.path.normcase(os.path.abspath(p)) != os.path.normcase(SRC)]
sys.path.insert(0, SRC)
SOURCES_SHA256 = {f: sha256_file(os.path.join(SRC, f)) for f in LIBRARY_FILES}
library = pd.DataFrame([{"file": f, "sha256": SOURCES_SHA256[f][:16],
                         "ok": SOURCES_SHA256[f] == FROZEN_FILE_SHA256[f] if f in FROZEN_FILE_SHA256 else True} for f in LIBRARY_FILES])
print("library"); print(library.to_string(index=False))
assert library["ok"].all(), "a pinned source file differs from its pin"

import set_attention_model, set_attention_cache, set_attention_data
import masked_concept_model as mcm
import mc_pilot_data as mpd
import mc_batched as mcb
import apriori.serialization as _ser
for mod in (set_attention_model, set_attention_cache, set_attention_data, mcm, mpd, mcb, _ser):
    assert os.path.normcase(os.path.abspath(mod.__file__)).startswith(os.path.normcase(SRC) + os.sep), f"{mod.__name__} from {mod.__file__}"
from set_attention_model import signed_log, subset_attention_levels_exact_bwd
from apriori.serialization import load_from_pickle, save_to_pickle

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
assert torch.cuda.is_available()
dev, f32 = "cuda", torch.float32
ENVIRONMENT = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
               "python": platform.python_version(), "os": platform.platform(),
               "tf32_matmul": torch.backends.cuda.matmul.allow_tf32, "tf32_cudnn": torch.backends.cudnn.allow_tf32}
print("environment", json.dumps(ENVIRONMENT))
os.makedirs(RUN_DIR, exist_ok=True)

# ----- data identity: the four files, their hashes and the fingerprints stored inside them -----
pdat = load_from_pickle(PILOT)
cdat = load_from_pickle(CTXF)
DATA_SHA256 = {os.path.basename(p): sha256_file(p) for p in (PILOT, CTXF, EDGES, REMOVED)}
labels = {os.path.basename(PILOT): (pdat["fingerprints"]["facts"],
                                    pdat["fingerprints"]["facts"] == FACTS_FP and "D5" in pdat.get("deviation", {})),
          os.path.basename(CTXF): (cdat["fingerprint"], cdat["fingerprint"] == CONTEXTS_FP and cdat["pilot_facts_fp"] == FACTS_FP),
          os.path.basename(EDGES): ("file sha256", DATA_SHA256[os.path.basename(EDGES)][:16].upper() == EDGES_SHA16)}
drows = []
for p in (PILOT, CTXF, EDGES, REMOVED):
    name = os.path.basename(p)
    r = {"file": name, "sha16": DATA_SHA256[name][:16].upper(), "mib": round(os.path.getsize(p) / 2 ** 20, 1)}
    if name in labels:
        r["stored_label"], r["label_ok"] = labels[name]
    drows.append(r)
data_check = pd.DataFrame(drows)
print("data"); print(data_check.to_string(index=False))
assert data_check["label_ok"].dropna().astype(bool).all(), "a stored fingerprint differs from the recorded identity"
decile_edges = load_from_pickle(EDGES)

# ----- derived arrays and counts -----
indptr, indices, data = pdat["indptr"], pdat["indices"], pdat["data"]
split = np.asarray(pdat["split"])
tr_idx, ho_idx = np.flatnonzero(split == "train"), np.flatnonzero(split == "holdout")
len_a = np.diff(indptr)
NT = torch.from_numpy(pdat["name_table"].astype(np.float32)).to(dev)
ctx12 = cdat["ctx"][N_T]
doc_freq = cdat["doc_freq_train"]
sl = signed_log(torch.as_tensor(np.asarray(data, dtype=np.float32))).numpy().astype(np.float64)
fact_row = np.repeat(np.arange(len(split)), len_a)
fact_local = np.arange(len(indices)) - indptr[fact_row]
tr_fact = np.isin(fact_row, tr_idx)
occ_filing, occ_local, occ_concept = fact_row[tr_fact], fact_local[tr_fact], indices[tr_fact]
N_OCC = len(occ_filing)

def epoch_keys(e):
    plan = mpd.column_epoch_plan(occ_concept, e, SEED, BATCH)
    order = np.random.default_rng((SEED, e, ORDER_SEED_TAG)).permutation(len(plan))
    plan = [plan[i] for i in order]
    allocc = np.concatenate([occ for _, occ in plan])
    assert len(allocc) == N_OCC and np.array_equal(np.sort(allocc), np.arange(N_OCC)), "plan coverage"
    return [[(int(occ_filing[o]), e, 1, (int(occ_local[o]),)) for o in occ] for _, occ in plan]

MICROBATCHES_PER_EPOCH = len(epoch_keys(0))
ho_rows = np.concatenate([np.arange(int(indptr[i]), int(indptr[i + 1])) for i in ho_idx])
counts = pd.DataFrame([
    {"count": "training filings", "value": len(tr_idx), "record": np.nan},
    {"count": "held out filings", "value": len(ho_idx), "record": np.nan},
    {"count": "training occurrences", "value": N_OCC, "record": RECORD["training occurrences"]},
    {"count": "held out targets", "value": len(ho_rows), "record": RECORD["held out targets"]},
    {"count": "microbatches per epoch", "value": MICROBATCHES_PER_EPOCH, "record": RECORD["microbatches per epoch"]},
])
counts["ok"] = counts["record"].isna() | counts["value"].eq(counts["record"])
print("counts"); print(counts.to_string(index=False))
assert counts["ok"].all(), "a derived count differs from the recorded counts"
counts

library
                   file           sha256   ok
       set_attention.py 8580eb3ce583edec True
  set_attention_data.py fd4c5a486477ccc3 True
 set_attention_model.py d7a7e72a2a111bb4 True
 set_attention_bench.py 80b9a7eddb88ddee True
masked_concept_model.py f6b49be2e07791e1 True
          mc_batched.py 571185cc24b467ee True
 set_attention_cache.py 86d8f308bbbc1a79 True
       mc_pilot_data.py 8bddded271636eda True
 mc_walk_jump_custom.py 385bfec34ad32cb8 True
environment {"gpu": "NVIDIA GeForce RTX 2080 SUPER", "torch": "2.5.1+cu121", "cuda": "12.1", "python": "3.12.7", "os": "Windows-10-10.0.19045-SP0", "tf32_matmul": false, "tf32_cudnn": false}
data
                             file            sha16  mib     stored_label label_ok
        mc3_pilot_data_d5.pkl.lz4 AA462904622BD717 92.2 1d6ee181e9fd1807     True
          mc3_contexts_d5.pkl.lz4 E5B25A32A6F99946 38.7 bb82e1d833c2983b     True
         mc3_decile_edges.pkl.lz4 F6270354A9B66881  0.0      file sha256     True
mc3_pilo

,count,value,record,ok
0,training filings,9997,NaN,True
1,held out filings,998,NaN,True
2,training occurrences,1234902,1234902.0,True
3,held out targets,136446,136446.0,True
4,microbatches per epoch,126767,126767.0,True


## Cell 3. Agreement gate: the shared cache against the reference traversal

Training does not start until this cell prints `GATE PASSED`. The cell takes the first accumulation group of epoch 1 (microbatches in plan order until 32 or more targets) and runs W two ways from the same certified initialization (`88071b7a2e52af9f`). The reference is pointer jumping under autograd, implementation (c) of Section 5, the pilot's own traversal in `mc_batched`. The other path is `walk="cache"`: the shared cache, `subset_attention_levels_exact_bwd` with `return_lse=True`, the implementation Section 7 measures. W items are built with `structures="both"`, so both paths read the same subset draw. `structures="v1"` is the library's name for the reference layout, one trie per item with the walk plan.

For each target, the reference walk rejects any subset that contains the target. The cache path removes those subsets before it builds the trie. The gate checks that both paths keep exactly the same subsets, in the same order, and that `P` (the drawn count) is unchanged in the Horvitz Thompson denominator. Gradients accumulate as in training: each microbatch's loss is weighted by its target count.

Tolerances: outputs and losses must satisfy max abs ≤ 1e-6 + 1e-4 × max|reference|. Each parameter tensor's gradient must have relative L2 ≤ 1e-3, or absolute L2 ≤ 1e-6 when the reference is zero. Both gradient lists must have the same length and the same tensor names. Any nonfinite loss, prediction, pooled output, L or gradient, on either path, fails its row. Both paths run in float32, with TF32 off.

Printed labels, in order:

- `group`: arm, `ref_walk`, `microbatches` and `targets` in the group.
- `identity`: forward only, summed over the group's microbatches.
  - `subsets_ref`: subsets in the reference tries. `usable_ref`: those the reference walk marks usable.
  - `subsets_cache`: subsets in the cache path's tries. `targets_without_subsets`: targets whose every drawn subset contains the target.
  - `usable_rule_ok`: the reference walk's usable mask equals "does not contain the target".
  - `same_subsets_same_order`: the cache path's subsets, as (target, sorted fact positions), equal the reference usable subsets in the same order.
  - `owner_ok`: every row's owner from `cache_subsets` equals its target. `k_ok`: every row's k equals its subset size and the reference k.
  - `finite`: the usable pooled outputs and L are finite on both paths.
  - `out_max_abs`, `out_err_over_tol`: pooled outputs against the reference walk, and the worst error divided by its tolerance (must be ≤ 1). `L_max_abs`, `L_err_over_tol`: the same for the log normalizer L.
  - `ok`: every microbatch passes all of the above.
- `training path`: `batch_loss` forward and backward over the whole group, accumulated as in training (`loss × targets`).
  - `loss_sum_ref`, `loss_sum_cache`, `loss_abs_diff`, `loss_tol`: the accumulated loss on each path, their difference, and its tolerance.
  - `exposures_ref`, `exposures_cache`: targets plus usable subsets. They must be equal.
  - `pred_sub_rows`: subset predictions returned by each path. `pred_full_max_abs` and `pred_sub_max_abs`: full context and subset predictions against the reference. `pred_err_over_tol` is the worst of the two divided by its tolerance.
  - `grad_worst_rel`: the largest relative L2 gradient difference over parameter tensors. `grad_err_over_tol`: the worst ratio to the gradient tolerance (must be ≤ 1). `worst_tensor`: the tensor where that ratio is largest.
  - `finite_ref`, `finite_cache`: every loss, prediction and gradient on that path is finite.
  - `ok`: all of the above hold.
  - Diagnostics, not gated and not an isolated performance comparison:
    - `ref_exec_s`, `cache_exec_s`: wall time from a synchronize before the group's first forward to a synchronize after its last backward. Each path runs alone on the GPU, because the other path's results have been moved to CPU first. The time includes the host reads inside `batch_loss` itself. Loss reads and gradient capture come after the timed span.
    - `ref_peak_gib`, `cache_peak_gib`: peak allocated memory over that span, measured from a cleared cache.
- `empty subsets`: two microbatches built from the group's largest microbatch. Each altered target's tries keep only the drawn subsets that contain it, so the reference walk finds none usable and the cache path gets no trie.
  - `mixed`: one target altered, the rest unchanged. `all empty`: every target altered, so the cache forest is empty.
  - Columns: `targets`, `empty_targets`, `loss_ref`, `loss_cache`, `loss_abs_diff`, and `expected_loss` (`all empty` only: 0.5 × mean full context Huber, a subset term of zero).
  - Then `exposures_ref`, `exposures_cache`, `cache_returns_none` (`all empty` only: `cache_subsets` returns None), `grad_err_over_tol`, `finite_ref`, `finite_cache` and `ok`.
- `preparation`: serial item construction for the group's keys, the median of 3 passes, in ms per microbatch.
  - Rows: `v1` (the reference layout, trie and `mc_plan`), `cache` (one trie per target without the target) and `both`.
  - `same_draw`: every structure was built from the same subset draw. The tries and `P` agree across the three layouts.ree across `v1`, `cache` and `both`.

In [ ]:
# Estimated run time: 1 to 3 minutes on the 2080 (first accumulation group of epoch 1 for W: identity on both
# paths, two empty subset cases, loss and gradients on both paths, serial preparation timing). Runs after cell 2, same kernel.
import torch.nn.functional as F
from set_attention_cache import build_prefix_trie_canonical

INIT_SHA = "88071b7a2e52af9f"
REF_WALK = {"W": "jump"}                         # the reference traversal, pointer jumping, implementation (c)
OUT_ATOL, OUT_RTOL = 1e-6, 1e-4                 # outputs and losses: max abs <= OUT_ATOL + OUT_RTOL x max|reference|
G_RTOL, G_ATOL = 1e-3, 1e-6                     # gradients: relative L2 <= G_RTOL, or L2 <= G_ATOL against a zero reference
PREP_REPS = 3

def sha16(b):
    return hashlib.sha256(b).hexdigest()[:16]

def new_model():
    torch.manual_seed(SEED)
    m = mcm.MaskedConceptModel(mcm.MCConfig()).to(dev, f32)
    h = sha16(b"".join(v.detach().cpu().numpy().tobytes() for v in m.state_dict().values()))
    assert h == INIT_SHA, f"initial state hash {h} differs from the certified {INIT_SHA}"
    m.train()
    return m

def finite(*ts):
    return all(bool(torch.isfinite(t).all()) for t in ts)

def out_tol(ref):
    return OUT_ATOL + OUT_RTOL * (float(ref.abs().max()) if ref.numel() else 0.0)

def max_abs(a, b):
    return float((a - b).abs().max()) if a.numel() else 0.0

def terminal_sets(tr):
    """Each subset of a trie as its sorted position tuple, in terminal order. The canonical build numbers every parent
    before its children, so one pass gives every node's path."""
    pos, par = tr["pos"], tr["parent"]
    path = [None] * len(pos)
    for i in range(len(pos)):
        p = int(par[i])
        path[i] = (path[p] if p >= 0 else ()) + (int(pos[i]),)
    return [path[int(t)] for t in tr["terminals"]]

def batch_inputs(model, items):
    """The inputs batch_loss forms: every filing's rows concatenated, one channel per target."""
    ns = np.array([it["n"] for it in items], dtype=np.int64)
    offs = np.concatenate([[0], np.cumsum(ns)[:-1]])
    cols = torch.as_tensor(np.concatenate([it["cols"] for it in items]), device=dev)
    vals = torch.as_tensor(np.concatenate([it["vals"] for it in items]), device=dev).to(NT.dtype)
    nv = NT[cols]
    ch_item = np.concatenate([np.full(len(it["tgt"]), b) for b, it in enumerate(items)])
    ch_j = np.concatenate([np.arange(len(it["tgt"])) for it in items])
    ch_local = np.concatenate([np.asarray(it["tgt"], dtype=np.int64) for it in items])
    ch_global = offs[ch_item] + ch_local
    tg = torch.as_tensor(ch_global, device=dev)
    z, V, _ = model.scores(nv, vals, nv[tg])
    return ns, offs, ch_item, ch_j, ch_local, ch_global, z, V, signed_log(vals)[tg]

# ----- the group, the datasets, and one subset draw across structures -----
keys0 = epoch_keys(0)
group_keys, acc = [], 0
for mb in keys0:
    group_keys.append(mb); acc += len(mb)
    if acc >= MIN_TARGETS:
        break
mk = lambda kind, s="v1": mpd.PilotDataset(indptr, indices, data, pdat["file_names"], ctx12, kind, M=M_W, run_factor=RUN_FACTOR,
                                           seed=SEED, rate=MASK_RATE, structures=s)
ds_W = {s: mk("W", s) for s in mpd.STRUCTURES}

def same_trie(a, b):
    return all(np.array_equal(np.asarray(a[f]), np.asarray(b[f])) for f in ("pos", "parent", "depth", "terminals"))

w_items, same_draw = [], True
for mb in group_keys:
    wi = []
    for key in mb:
        a, c, b = ds_W["v1"][key], ds_W["cache"][key], ds_W["both"][key]
        same_draw &= a["P"] == b["P"] == c["P"] and same_trie(a["trie"], b["trie"])
        same_draw &= all((x is None and y is None) or (x is not None and y is not None and same_trie(x, y))
                         for x, y in zip(b["tries_excl"], c["tries_excl"]))
        assert b["i"] == key[0] and list(b["tgt"]) == list(key[3]), "item differs from its key"
        wi.append(b)
    w_items.append(wi)
assert same_draw, "the W structures were not built from the same subset draw"
ARMS = (("W", w_items),)
grp = pd.DataFrame([{"arm": k, "ref_walk": REF_WALK[k], "microbatches": len(il), "targets": sum(len(it["tgt"]) for its in il for it in its)}
                    for k, il in ARMS])
print("group"); print(grp.to_string(index=False))

# ----- identity: subsets, owners, sizes, pooled outputs and L against the reference walk (forward only) -----
@torch.no_grad()
def identity_check(model, kind, items):
    ns, offs, ch_item, ch_j, ch_local, ch_global, z, V, _ = batch_inputs(model, items)
    C = len(ch_global)
    plan_fn, walk_fn = mcb.WALKS[REF_WALK[kind]]
    ref_sets, cache_keys, blocks = [], [], []
    for c in range(C):
        b, j = int(ch_item[c]), int(ch_j[c])
        ref_sets.append(terminal_sets(items[b]["trie"]))
        tr = items[b]["tries_excl"][j]
        cache_keys += [] if tr is None else [(c, S) for S in terminal_sets(tr)]
        blocks.append(None if tr is None else (offs[b] + np.arange(ns[b]), tr))
    tries = [mcb._shift(items[int(ch_item[c])]["trie"], offs[int(ch_item[c])]) for c in range(C)]
    pool, L, k, use = walk_fn(z, V, plan_fn(tries), owned=True, owner_target_pos=ch_global)
    all_keys = [(c, S) for c in range(C) for S in ref_sets[c]]
    expected = np.array([int(ch_local[c]) not in S for c, S in all_keys])
    u = use[:, 0].cpu().numpy().astype(bool)
    ref_keys = [kk for kk, ok in zip(all_keys, u) if ok]
    ui = torch.as_tensor(u, device=dev)
    pr, Lr, kr = pool[ui, 0], L[ui, 0], k[ui, 0]
    r = {"subsets_ref": len(all_keys), "usable_ref": int(u.sum()), "subsets_cache": len(cache_keys),
         "targets_without_subsets": sum(blk is None for blk in blocks),
         "usable_rule_ok": bool(np.array_equal(u, expected)), "same_subsets_same_order": ref_keys == cache_keys,
         "owner_ok": False, "k_ok": False, "finite": finite(pr, Lr),
         "out_max_abs": np.nan, "out_err_over_tol": np.nan, "L_max_abs": np.nan, "L_err_over_tol": np.nan}
    res = mcb.cache_subsets(z, V, blocks, dev)
    if res is None:
        ok0 = len(cache_keys) == 0 and len(ref_keys) == 0
        r.update({"owner_ok": ok0, "k_ok": ok0, "out_max_abs": 0.0, "out_err_over_tol": 0.0, "L_max_abs": 0.0, "L_err_over_tol": 0.0})
    else:
        out, Lc, owner, kc = res
        r["finite"] = r["finite"] and finite(out, Lc)
        if r["same_subsets_same_order"] and out.shape[0] == len(cache_keys) and r["finite"]:
            r["owner_ok"] = owner.cpu().numpy().tolist() == [c for c, _ in cache_keys]
            r["k_ok"] = (kc.cpu().numpy().astype(np.int64).tolist() == [len(S) for _, S in cache_keys]
                         and bool(torch.equal(kc, kr.to(kc.dtype))))
            r["out_max_abs"], r["L_max_abs"] = max_abs(out, pr), max_abs(Lc, Lr)
            r["out_err_over_tol"], r["L_err_over_tol"] = r["out_max_abs"] / out_tol(pr), r["L_max_abs"] / out_tol(Lr)
    r["ok"] = bool(r["usable_rule_ok"] and r["same_subsets_same_order"] and r["owner_ok"] and r["k_ok"] and r["finite"]
                   and r["out_err_over_tol"] <= 1 and r["L_err_over_tol"] <= 1)
    return r

id_rows = []
for kind, il in ARMS:
    model = new_model()
    rs = [identity_check(model, kind, items) for items in il]
    del model
    row = {"arm": kind, **{f: sum(x[f] for x in rs) for f in ("subsets_ref", "usable_ref", "subsets_cache", "targets_without_subsets")},
           **{f: all(x[f] for x in rs) for f in ("usable_rule_ok", "same_subsets_same_order", "owner_ok", "k_ok", "finite")},
           **{f: float(np.max([x[f] for x in rs])) for f in ("out_max_abs", "out_err_over_tol", "L_max_abs", "L_err_over_tol")},
           "ok": all(x["ok"] for x in rs)}
    id_rows.append(row)
identity = pd.DataFrame(id_rows)
print("identity"); print(identity.to_string(index=False))
torch.cuda.empty_cache()

# ----- training path helpers -----
def run_path(kind, il, walk):
    """Forward and backward of the group, accumulated by target count. Execution is timed between synchronizes; loss
    reads and gradient capture follow the timed span, and every captured result is moved to CPU."""
    model = new_model()
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache(); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    losses, counts, expo, pf, ps = [], [], 0, [], []
    t0 = time.perf_counter()
    for items in il:
        n = sum(len(it["tgt"]) for it in items)
        loss, e_, pfull, psub = mcb.batch_loss(model, kind, items, NT, dev, walk=walk)
        (loss * n).backward()
        losses.append(loss.detach()); counts.append(n); expo += int(e_)
        pf.append(pfull.detach()); ps.append(psub.detach())
    torch.cuda.synchronize()
    seconds, peak = time.perf_counter() - t0, torch.cuda.max_memory_allocated() / 2**30
    loss_t = torch.stack(losses).double().cpu()
    out = {"loss_sum": float((loss_t * torch.tensor(counts, dtype=torch.float64)).sum()), "exposures": expo,
           "pred_full": torch.cat(pf).cpu(), "pred_sub": torch.cat(ps).cpu(),
           "grads": [(nm, None if p.grad is None else p.grad.detach().double().cpu()) for nm, p in model.named_parameters()],
           "seconds": seconds, "peak_gib": peak}
    out["finite"] = finite(loss_t, out["pred_full"], out["pred_sub"], *[g for _, g in out["grads"] if g is not None])
    del model, losses, pf, ps
    torch.cuda.empty_cache()
    return out

def grad_compare(ga, gb):
    """(worst relative L2, worst ratio to tolerance, tensor of the worst ratio). Missing, mismatched or nonfinite
    gradients return an infinite ratio."""
    assert len(ga) == len(gb), f"gradient lists differ in length: {len(ga)} and {len(gb)}"
    worst_rel, worst_score, worst_name = 0.0, 0.0, ""
    for (na, a), (nb, b) in zip(ga, gb):
        assert na == nb, f"gradient names differ: {na} and {nb}"
        if (a is None) != (b is None):
            return float("inf"), float("inf"), na
        if a is None:
            continue
        if not finite(a, b):
            return float("inf"), float("inf"), na
        ra, d = float(a.norm()), float((a - b).norm())
        score = d / (G_RTOL * ra) if ra > 0 else d / G_ATOL
        if ra > 0:
            worst_rel = max(worst_rel, d / ra)
        if score > worst_score:
            worst_score, worst_name = score, na
    return worst_rel, worst_score, worst_name

# ----- empty subsets (W): one target without usable subsets, then a microbatch with none -----
def target_only(it):
    """Copy of a W item whose tries keep only the drawn subsets that contain its target (None when there are none)."""
    assert len(it["tgt"]) == 1, "column schedule items carry one target"
    t = int(it["tgt"][0])
    kept = [S for S in terminal_sets(it["trie"]) if t in S]
    if not kept:
        return None
    m = {k: v for k, v in it.items() if k != "plan"}
    m["trie"], m["tries_excl"] = build_prefix_trie_canonical(kept), [None]
    return m

base = max(w_items, key=len)
alt = [target_only(it) for it in base]
first = next(i for i, a in enumerate(alt) if a is not None)
mixed = [alt[i] if i == first else it for i, it in enumerate(base)]
all_empty = [a for a in alt if a is not None]
assert len(mixed) >= 2 and sum(it["tries_excl"][0] is not None for it in mixed) >= 1, "mixed case needs targets with subsets"
assert len(all_empty) >= 1
em_rows = []
for case, items in (("mixed", mixed), ("all empty", all_empty)):
    a, b = run_path("W", [items], REF_WALK["W"]), run_path("W", [items], "cache")
    _, g_score, _ = grad_compare(a["grads"], b["grads"])
    n = sum(len(it["tgt"]) for it in items)
    la, lb = a["loss_sum"] / n, b["loss_sum"] / n
    row = {"case": case, "targets": n, "empty_targets": sum(it["tries_excl"][0] is None for it in items),
           "loss_ref": la, "loss_cache": lb, "loss_abs_diff": abs(la - lb), "expected_loss": np.nan,
           "exposures_ref": a["exposures"], "exposures_cache": b["exposures"], "cache_returns_none": np.nan,
           "grad_err_over_tol": g_score, "finite_ref": a["finite"], "finite_cache": b["finite"]}
    ok = (a["finite"] and b["finite"] and row["loss_abs_diff"] <= OUT_ATOL + OUT_RTOL * abs(la)
          and a["exposures"] == b["exposures"] and g_score <= 1)
    if case == "all empty":
        y = signed_log(torch.as_tensor(np.array([it["vals"][int(it["tgt"][0])] for it in items], dtype=np.float32)))
        row["expected_loss"] = float((0.5 * F.huber_loss(a["pred_full"], y, reduction="none")).mean())
        row["cache_returns_none"] = mcb.cache_subsets(None, None, [None] * len(items), dev) is None
        ok = (ok and np.isfinite(row["expected_loss"]) and abs(lb - row["expected_loss"]) <= OUT_ATOL + OUT_RTOL * abs(row["expected_loss"])
              and b["exposures"] == n and b["pred_sub"].numel() == 0 and row["cache_returns_none"])
    row["ok"] = bool(ok)
    em_rows.append(row)
    del a, b
empty = pd.DataFrame(em_rows)

# ----- training path: the whole group, accumulated as in training -----
tp_rows = []
for kind, il in ARMS:
    a = run_path(kind, il, REF_WALK[kind])
    b = run_path(kind, il, "cache")
    g_rel, g_score, g_name = grad_compare(a["grads"], b["grads"])
    same_rows = a["pred_sub"].shape == b["pred_sub"].shape
    pfe = max_abs(a["pred_full"], b["pred_full"])
    pse = max_abs(a["pred_sub"], b["pred_sub"]) if same_rows else np.nan
    pred_err = float(np.max([pfe / out_tol(a["pred_full"]), pse / out_tol(a["pred_sub"]) if same_rows else np.nan]))
    row = {"arm": kind, "loss_sum_ref": a["loss_sum"], "loss_sum_cache": b["loss_sum"], "loss_abs_diff": abs(a["loss_sum"] - b["loss_sum"]),
           "loss_tol": OUT_ATOL + OUT_RTOL * abs(a["loss_sum"]), "exposures_ref": a["exposures"], "exposures_cache": b["exposures"],
           "pred_sub_rows": f"{a['pred_sub'].numel()} / {b['pred_sub'].numel()}", "pred_full_max_abs": pfe, "pred_sub_max_abs": pse,
           "pred_err_over_tol": pred_err, "grad_worst_rel": g_rel, "grad_err_over_tol": g_score, "worst_tensor": g_name,
           "finite_ref": a["finite"], "finite_cache": b["finite"],
           "ref_exec_s": a["seconds"], "cache_exec_s": b["seconds"], "ref_peak_gib": a["peak_gib"], "cache_peak_gib": b["peak_gib"]}
    row["ok"] = bool(a["finite"] and b["finite"] and np.isfinite(row["loss_abs_diff"]) and row["loss_abs_diff"] <= row["loss_tol"]
                     and a["exposures"] == b["exposures"] and same_rows and pred_err <= 1 and g_score <= 1)
    tp_rows.append(row)
    del a, b
training = pd.DataFrame(tp_rows)
print("training path"); print(training.to_string(index=False))
print("empty subsets"); print(empty.to_string(index=False))

# ----- preparation: serial item construction for the group's keys -----
def prep_ms(ds):
    times = []
    for _ in range(PREP_REPS):
        t0 = time.perf_counter()
        for mb in group_keys:
            for key in mb:
                ds[key]
        times.append(time.perf_counter() - t0)
    return 1e3 * float(np.median(times)) / len(group_keys)

prep = pd.DataFrame([{"arm": "W", "structure": s, "ms_per_microbatch": prep_ms(ds_W[s]), "same_draw": same_draw} for s in mpd.STRUCTURES])
print("preparation"); print(prep.to_string(index=False))

assert identity["ok"].all(), "identity check failed"
assert training["ok"].all(), "training path disagreement"
assert empty["ok"].all(), "empty subset case failed"
print("GATE PASSED")

## Cell 4. Gate record

Saves cell 3's output with the library identity it ran under, to `E:\apriori-ml-v3\pilot\runs\v3_gate.pkl.lz4`. Before saving, the cell recomputes the full sha256 of every library file and checks the pinned ones against their pins. That verifies the files on disk still match. That the loaded functions came from those bytes rests on cell 2's fresh kernel import checks: no library or pilot module was imported beforehand, and every module loaded from `E:\apriori-ml-v3\src`. This library is the one the three training arms and the cost study use.

The record is written to a temporary file and then moved into place, so a partially written record never takes the final name. It is then read back. The source hash dictionary and the five tables must equal what is in memory. The cell does not overwrite an existing record.

The record holds:

- the gate tables: `group`, `identity`, `training path`, `empty subsets`, `preparation`
- the group's keys
- the tolerances and reference walk
- the settings
- the environment
- the full source hashes of the nine library files
- the initialization hash
- the time it was written

The timings and peak memory in `training path` and `preparation` are stored as single group diagnostics. The training runs establish training time and memory.

Printed labels, in order:

- `sources`: each library file, its full sha256, and `ok` (True for a pinned file that matches its pin, True by definition for a recorded file).
- `saved`: the record's path, its size in KiB, the first 16 characters of the file's sha256, and `read_back_ok` when the reloaded source hashes and the five reloaded tables equal those in memory.

In [3]:
# Estimated run time: under 10 seconds. Runs after cell 3, same kernel.
import datetime

GATE_RECORD = os.path.join(RUN_DIR, "v3_gate.pkl.lz4")
assert not os.path.exists(GATE_RECORD), f"{GATE_RECORD} exists; the gate record is written once"
assert identity["ok"].all() and training["ok"].all() and empty["ok"].all(), "cell 3 did not pass"

src = pd.DataFrame([{"file": f, "sha256": sha256_file(os.path.join(SRC, f))} for f in LIBRARY_FILES])
src["ok"] = [h == FROZEN_FILE_SHA256[f] if f in FROZEN_FILE_SHA256 else True for f, h in zip(src["file"], src["sha256"])]
print("sources"); print(src.to_string(index=False))
assert src["ok"].all(), "a pinned source file on disk differs from its pin"
assert dict(zip(src["file"], src["sha256"])) == SOURCES_SHA256, "a source file changed since cell 2 hashed it"

gate_tables = {"group": grp, "identity": identity, "training path": training, "empty subsets": empty, "preparation": prep}
gate_record = {
    "written": datetime.datetime.now().isoformat(timespec="seconds"),
    "sources_sha256": dict(zip(src["file"], src["sha256"])),
    "environment": ENVIRONMENT,
    "init_state_sha16": INIT_SHA,
    "ref_walk": REF_WALK,
    "tolerances": {"out_atol": OUT_ATOL, "out_rtol": OUT_RTOL, "grad_rtol": G_RTOL, "grad_atol": G_ATOL},
    "settings": {"M_W": M_W, "run_factor": RUN_FACTOR, "N_T": N_T, "batch": BATCH, "order_seed_tag": ORDER_SEED_TAG,
                 "min_targets": MIN_TARGETS, "seed": SEED, "mask_rate": MASK_RATE},
    "group_keys": group_keys,
    "tables": gate_tables,
    "diagnostics_note": "exec seconds, peak memory and preparation ms are single group diagnostics, not training measurements",
}
save_to_pickle(gate_record, GATE_RECORD, lz4_compression=True)   # atomic: save_to_pickle writes a temporary file in the same folder, then os.replace

back = load_from_pickle(GATE_RECORD)
read_back_ok = (back["sources_sha256"] == gate_record["sources_sha256"]
                and set(back["tables"]) == set(gate_tables)
                and all(back["tables"][k].equals(v) for k, v in gate_tables.items()))
print("saved"); print(pd.DataFrame([{"path": GATE_RECORD, "kib": os.path.getsize(GATE_RECORD) / 1024,
                                     "sha16": sha256_file(GATE_RECORD)[:16], "read_back_ok": read_back_ok}]).to_string(index=False))
assert read_back_ok, "the saved gate record does not read back equal to the gate in memory"
del back

sources
                   file                                                           sha256   ok
       set_attention.py 8580eb3ce583edec900c993d58793e98d2ef4e9c434f0dddc5a58991918f0d52 True
  set_attention_data.py fd4c5a486477ccc30a7c202891ffbc579dc02b2bc5ae5988e94cbe6d7ac23cea True
 set_attention_model.py d7a7e72a2a111bb40e45cbc61ed844349e51ed15c7c22749a817893eb8eee345 True
 set_attention_bench.py 80b9a7eddb88ddee8948d997808643e476328e915b7ac09847e3ff25f64ea72a True
masked_concept_model.py f6b49be2e07791e1e8f39729f881e4488a85943f8de35a84ebf83071d751f330 True
          mc_batched.py 571185cc24b467ee1955561b19348522548e328ecd7b0aa1767b8e514d78c43e True
 set_attention_cache.py 86d8f308bbbc1a7934fde24fc65abd90f5e2ef4aeeabff9a5f9795d39db1ccff True
       mc_pilot_data.py 8bddded271636eda59db5b258ad3ba7607c0fb708e67b3597aa25bc62f0ff525 True
 mc_walk_jump_custom.py 385bfec34ad32cb80496c11205aa18ee02131777719dcdf6f03b9867db12a402 True
saved
                                       path   

## Cell 5. Training harness for the three arms

Defines the harness and checks its inputs; no training runs here. Settings are cell 2's. W computes its subset term through the shared cache (`walk="cache"`), the implementation cell 3 certified against the reference traversal.

| Arm | Batch kind | Update rule | Subset term | Evaluation |
|---|---|---|---|---|
| `W0_registered` | W0 | registered: one update per microbatch, learning rate by update count | none | full context |
| `W0` | W0 | D4: one update per 32 or more accumulated targets, learning rate by target exposure | none | full context |
| `W` | W | D4 | sampled subsets, Horvitz Thompson, shared cache | full context |

The ranked contexts file `mc3_contexts_d5.pkl.lz4` is loaded by cell 2 and hashed into every contract. No arm in this notebook trains or evaluates on it.

Checks, each asserted:

- Both epoch plans match their certified hashes (`c9ec2fc1ce2e0860`, `8aef1402fd7de8d8`).
- The held out target count equals the recorded 136,446.
- Every held out target falls in exactly one magnitude band and exactly one decile bin. The bin rule is `searchsorted(edges, |y|, side="left") − 1`, with the right edge inclusive.
- In training (cell 6):
  - every model starts from initialization `88071b7a2e52af9f`
  - the loader follows plan order
  - under D4, the optimizer steps at each epoch end equal the recorded 32,995 and 65,953
  - under the registered rule, they equal the microbatch count

Evaluation runs with the model in eval mode and with the RNG states saved. The incoming training flag and the CPU and CUDA RNG states are restored even if evaluation raises.

W accounting is exact on every microbatch:

- `P`: subsets drawn per filing.
- `usable`: subsets that do not contain the target.
- `zero targets`: targets with no usable subset.

Each arm writes its own files to `E:\apriori-ml-v3\pilot\runs`:

- `v3_{arm}_last.pt`: the checkpoint, which is authoritative and carries the curve.
- `v3_{arm}_epoch{1,2}.pt`: one checkpoint per epoch.
- `v3_{arm}_curve.pkl.lz4`: the curve.

The contract hashes everything that affects training or the reported evaluation:

- the full source sha256 of the nine library files, from cell 2
- the full sha256 of the four data files, from cell 2, and the decile edge hash
- the data fingerprints, the split and the model config
- the arm's parameters
- the update and evaluation settings: `MASK_RATE`, `MIN_TARGETS`, `WARMUP_FRAC`, `EVAL_CHUNK`, the probe (`N_PROBE`, filings of at most 1,024 facts, seed tags 181 and 171), `BOOT` and its seed, and `BANDS`
- the harness version

A checkpoint written under a different contract is refused. Times split training from evaluation, so cost per epoch can be read without evaluation. Cell 3's gate and preparation diagnostics stay separate from the training measurements.

Printed labels, in order:

- `plans`: per epoch, `plan_sha16`, `microbatches`, `targets` and `ok`.
- `held out`: `targets`, `filings`, `targets_equal_record`, `one_band_each` and `one_bin_each`.
- `bins`: per decile bin, its held out target count.
- `arms`: per arm, `kind`, `rule`, `walk`, `evaluation` and `contract`.ntract` and the v1.0 hours for two epochs (`v1_hours`).

In [4]:
# Estimated run time: 1 to 3 minutes (both epoch plans hashed, held out sets and probe tensors built). Defines the harness
# for the three arms; no training. Runs after cell 4, same kernel.
from dataclasses import asdict
from torch.utils.data import DataLoader

HARNESS_VERSION = "v3-pilot-1"
PLAN_SHA = {0: "c9ec2fc1ce2e0860", 1: "8aef1402fd7de8d8"}
STEPS_AT_EPOCH = {1: 32_995, 2: 65_953}                       # D4 optimizer steps at each epoch end, recorded
PROBE_MAX_FACTS, PROBE_FILING_TAG, PROBE_PERM_TAG, BOOT_TAG = 1024, 181, 171, 91
ARM_SPEC = {
    "W0_registered": {"kind": "W0", "rule": "registered", "walk": "levels", "ranked": False, "structures": "v1"},
    "W0":            {"kind": "W0", "rule": "D4",         "walk": "levels", "ranked": False, "structures": "v1"},
    "W":             {"kind": "W",  "rule": "D4",         "walk": "cache",  "ranked": False, "structures": "cache"},
}

def huber(pred, y):
    r = np.abs(pred - y)
    return np.where(r <= 1.0, 0.5 * r * r, r - 0.5)

# ----- epoch plans, certified -----
def plan_sha(keys):
    return sha16(np.concatenate([np.array([k[0] for k in mb] + [k[3][0] for k in mb]) for mb in keys]).astype(np.int64).tobytes())

KEYS, prow = {}, []
for e in range(EPOCHS_RUN):
    KEYS[e] = epoch_keys(e)
    h = plan_sha(KEYS[e])
    prow.append({"epoch": e + 1, "plan_sha16": h, "microbatches": len(KEYS[e]), "targets": sum(len(mb) for mb in KEYS[e]), "ok": h == PLAN_SHA[e]})
plans = pd.DataFrame(prow)
print("plans"); print(plans.to_string(index=False))
assert plans["ok"].all(), "an epoch plan differs from the certified plan"
BOUNDS = [int(round(MICROBATCHES_PER_EPOCH * k / 10)) for k in range(1, 11)]
WARM_U, TOTAL_U = int(round(WARMUP_FRAC * MICROBATCHES_PER_EPOCH)), MICROBATCHES_PER_EPOCH * EPOCHS_REGISTERED
WARM_T, TOTAL_T = WARMUP_FRAC * N_OCC, N_OCC * EPOCHS_REGISTERED

def lr_registered(g):
    return LR * (g + 1) / WARM_U if g < WARM_U else LR * 0.5 * (1 + math.cos(math.pi * min(g - WARM_U, TOTAL_U - WARM_U) / max(TOTAL_U - WARM_U, 1)))

def lr_d4(t):
    return LR * t / WARM_T if t < WARM_T else LR * 0.5 * (1 + math.cos(math.pi * min(t - WARM_T, TOTAL_T - WARM_T) / max(TOTAL_T - WARM_T, 1)))

# ----- held out targets; bands, groups, bins, probe -----
y_ho, c_ho, fid = sl[ho_rows], indices[ho_rows].astype(np.int64), fact_row[ho_rows]
ctx12a = np.asarray(ctx12)
ay = np.abs(y_ho)
bands = {nm: (ay >= lo) & (ay < hi) for lo, hi, nm in BANDS}
df_ho = doc_freq[c_ho]
groups = {"seen_common": df_ho >= 50, "seen_rare": (df_ho >= 1) & (df_ho < 50), "unseen": df_ho == 0}
N_BINS = int(decile_edges["n_bins"])
ho_bin = np.searchsorted(decile_edges["assign_edges_signed_log"], ay, side="left") - 1      # right edge inclusive
bins = {f"bin {b}": ho_bin == b for b in range(N_BINS)}
held = pd.DataFrame([{"targets": len(y_ho), "filings": len(np.unique(fid)),
                      "targets_equal_record": len(y_ho) == RECORD["held out targets"],
                      "one_band_each": bool((np.sum(list(bands.values()), axis=0) == 1).all()),
                      "one_bin_each": bool(((ho_bin >= 0) & (ho_bin < N_BINS)).all() and (np.sum(list(bins.values()), axis=0) == 1).all())}])
print("held out"); print(held.to_string(index=False))
bin_counts = pd.DataFrame([{"bin": k, "targets": int(m.sum())} for k, m in bins.items()])
print("bins"); print(bin_counts.to_string(index=False))
assert held[["targets_equal_record", "one_band_each", "one_bin_each"]].all(axis=None), "held out set check failed"

def filing_tensors(i):
    lo, hi = int(indptr[i]), int(indptr[i + 1])
    return (NT[torch.as_tensor(indices[lo:hi].astype(np.int64), device=dev)],
            torch.as_tensor(np.asarray(data[lo:hi], dtype=np.float32), device=dev),
            torch.as_tensor(ctx12a[lo:hi].astype(np.int64), device=dev))

probe_f = np.sort(np.random.default_rng((SEED, PROBE_FILING_TAG)).choice(ho_idx[len_a[ho_idx] <= PROBE_MAX_FACTS], N_PROBE, replace=False))
probe = []
for i in probe_f:
    nv, raw, ctx = filing_tensors(i)
    perm = torch.as_tensor(np.random.default_rng((SEED, PROBE_PERM_TAG, int(i))).permutation(nv.shape[0]), device=dev)
    probe.append((nv, raw, raw[perm], ctx))

@torch.no_grad()
def predict_filing(model, nv, vals, ctx, ranked):
    """Full context: every other fact of the filing. Ranked: the target's 12 ranked context facts. The target is never a member."""
    n = nv.shape[0]
    preds = []
    for s in range(0, n, EVAL_CHUNK):
        tg = torch.arange(s, min(n, s + EVAL_CHUNK), device=dev)
        c = torch.arange(len(tg), device=dev)
        z, V, e = model.scores(nv, vals, nv[tg])
        if ranked:
            member = torch.zeros(len(tg), n, dtype=torch.bool, device=dev)
            cr = ctx[tg]
            valid = cr >= 0
            member[c[:, None].expand_as(cr)[valid], cr[valid]] = True
            assert not bool(member[c, tg].any()), "a ranked context contains its target"
        else:
            member = torch.ones(len(tg), n, dtype=torch.bool, device=dev)
            member[c, tg] = False
        fp, fL, fk, fu = mcm.mc_masked_attention(z, V, member, c)
        assert bool(fu.all()), "empty context at evaluation"
        preds.append(model.predict(fp, fL, fk, e))
    return torch.cat(preds).double().cpu().numpy()

@torch.no_grad()
def evaluate(model, ranked):
    """Held out loss per target plus summaries. The incoming training flag and the CPU and CUDA RNG states are
    restored in finally, so they hold even if evaluation raises."""
    t0 = time.perf_counter()
    was_training = model.training
    rc, rg = torch.get_rng_state(), torch.cuda.get_rng_state_all()
    try:
        model.eval()
        pred = np.concatenate([predict_filing(model, *filing_tensors(i), ranked) for i in ho_idx])
        assert np.isfinite(pred).all(), "nonfinite held out prediction"
        dp, pb = [], []
        for nv, raw, shuf, ctx in probe:
            p0, p1 = predict_filing(model, nv, raw, ctx, ranked), predict_filing(model, nv, shuf, ctx, ranked)
            dp.append(np.abs(p1 - p0)); pb.append(p0)
    finally:
        model.train(was_training)
        torch.set_rng_state(rc); torch.cuda.set_rng_state_all(rg)
    l = huber(pred, y_ho)
    ev = {"heldout": float(l.mean()), **{k: float(l[m].mean()) if m.any() else float("nan") for k, m in bands.items()},
          **{k: float(l[m].mean()) for k, m in groups.items()},
          "d_pred_shuffle": float(np.median(np.concatenate(dp))), "pred_spread": float(np.std(np.concatenate(pb)))}
    return l, ev, time.perf_counter() - t0

def cluster_ci(d, f):
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    uniq, inv = np.unique(f, return_inverse=True)
    s, c = np.bincount(inv, weights=d), np.bincount(inv)
    idx = np.random.default_rng((SEED, BOOT_TAG)).integers(0, len(uniq), size=(BOOT, len(uniq)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return float(d.mean()), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

# ----- per arm parameters and contracts -----
def arm_params(arm):
    sp = ARM_SPEC[arm]
    p = {"arm": arm, "kind": sp["kind"], "rule": sp["rule"], "walk": sp["walk"], "structures": sp["structures"], "data": "D5", "order": "interleaved",
         "order_seed_tag": ORDER_SEED_TAG, "M_W": M_W, "run_factor": RUN_FACTOR, "N_T": N_T, "batch": BATCH, "mask_rate": MASK_RATE,
         "epochs_registered": EPOCHS_REGISTERED, "epochs_run": EPOCHS_RUN, "lr": LR, "wd": WD, "seed": SEED,
         "init_state_sha16": INIT_SHA, "plan_sha16": PLAN_SHA,
         "eval": "ranked context, 12 facts, target excluded" if sp["ranked"] else "full context, leave one out",
         "eval_settings": {"eval_chunk": EVAL_CHUNK, "n_probe": N_PROBE, "probe_max_facts": PROBE_MAX_FACTS,
                           "probe_seeds": [[SEED, PROBE_FILING_TAG], [SEED, PROBE_PERM_TAG, "filing"]],
                           "boot": BOOT, "boot_seed": [SEED, BOOT_TAG], "bands": [[lo, str(hi), nm] for lo, hi, nm in BANDS],
                           "bin_rule": "searchsorted(assign_edges_signed_log, |y|, left) - 1"}}
    p["schedule"] = ({"warmup_frac": WARMUP_FRAC, "warmup_updates": WARM_U, "total_updates": TOTAL_U, "lr_by": "update count"} if sp["rule"] == "registered" else
                     {"min_targets_per_step": MIN_TARGETS, "warmup_frac": WARMUP_FRAC, "warmup_targets": WARM_T, "total_targets": TOTAL_T,
                      "lr_by": "target exposure, group end"})
    return p

def arm_contract(arm):
    return sha16(json.dumps({"sources": SOURCES_SHA256, "data_files": DATA_SHA256, "edges_sha16": EDGES_SHA16,
                             "facts_fp": FACTS_FP, "contexts_fp": CONTEXTS_FP, "split": sha16(split.astype("U").tobytes()),
                             "mc_config": asdict(mcm.MCConfig()), "params": arm_params(arm), "harness_version": HARNESS_VERSION},
                            sort_keys=True, default=str).encode())

arms = pd.DataFrame([{"arm": a, "kind": s["kind"], "rule": s["rule"], "walk": s["walk"] if s["kind"] in ("W", "T") else "none",
                      "evaluation": "ranked" if s["ranked"] else "full", "contract": arm_contract(a)}
                     for a, s in ARM_SPEC.items()])
print("arms"); print(arms.to_string(index=False))

plans
 epoch       plan_sha16  microbatches  targets   ok
     1 c9ec2fc1ce2e0860        126767  1234902 True
     2 8aef1402fd7de8d8        126767  1234902 True
held out
 targets  filings  targets_equal_record  one_band_each  one_bin_each
  136446      998                  True           True          True
bins
  bin  targets
bin 0    15290
bin 1    16274
bin 2    15752
bin 3    14774
bin 4    13591
bin 5    12921
bin 6    12673
bin 7    12416
bin 8    11816
bin 9    10939
arms
          arm kind       rule  walk evaluation         contract
W0_registered   W0 registered  none       full de1c787361806f25
           W0   W0         D4  none       full 235932e5fe71eb60
            W    W         D4 cache       full 3e697e75dd41d069


## Cell 6. Training: the three arms

Runs `W0_registered`, `W0` and `W` in that order, two epochs each, with cell 5's harness. Every arm starts from initialization `88071b7a2e52af9f` and follows the certified epoch plans. W computes its subset term through the shared cache. W items carry one trie per target, built without that target (`structures="cache"`). The cell can be resumed: rerun it after any interruption and each arm continues from `v3_{arm}_last.pt`. An arm with both epochs complete is skipped. Expect about 8 hours on the 2080 for the three arms from initialization, including the 21 evaluations per arm.

**Update rules.**

- `registered`: `loss.backward()` and one AdamW step per microbatch. The learning rate depends on the update count: linear warmup over 10% of one epoch's microbatches, then cosine over 8 epochs.
- `D4`: `(loss × targets).backward()` for each microbatch. The gradients are divided by the accumulated target count, and one AdamW step is taken once 32 or more targets have accumulated, or at epoch end. The learning rate depends on target exposure at the end of the group: linear warmup over 10% of one epoch's targets, then cosine over 8 epochs' targets.
- Both rules refuse to step on a nonfinite loss or gradient.
- Each step also records diagnostics: a parameter clone, the float64 gradient norm and the float64 update norm.

**Asserted in training:**

- The loader returns the planned filings and targets in plan order.
- At each epoch end, the optimizer step count equals 32,995 and 65,953 under D4, or the microbatch count under the registered rule.

**Reproducibility on resume.** Each loader rebuild gets its own generator, seeded from (seed, arm, epoch, starting microbatch). Worker seeds never draw on the global RNG, so a resume or a rebuild after a stall does not change training randomness. The dataset's subset draws and targets are keyed by filing and epoch, so they don't depend on the loader either.

**Checkpoints.**

- One is written at every 10% boundary if no accumulation group is open; otherwise it is written at the next optimizer step. Partial gradients are never stored.
- An epoch checkpoint is written at each epoch end.
- Every checkpoint goes to a temporary file and is then moved into place.
- The curve is written next to every checkpoint through `save_to_pickle`, which writes a temporary file in the same folder and then calls `os.replace`.

**Timing and memory.**

- `train_h`: training wall time. It includes loader waits and the per step diagnostics above, and excludes evaluation and checkpoint writes. It measures this training protocol, not bare execution of the cache.
- `eval_h`: evaluation wall time.
- `peak_gib`: peak allocated memory during training, including the diagnostics' parameter clone. The peak counter is reset after every evaluation, so evaluation is excluded.
- Cell 3's diagnostics are not mixed in.

Printed per arm at the start: either `init` with the initial held out loss, or `resumed` with the epoch, microbatch and step count.

Printed at each 10% boundary, in order:

- `arm`, `epoch`, `pct`: where the run is.
- `steps`: optimizer steps so far.
- `heldout`: mean held out Huber loss under full context evaluation.
- `0 to 1`, `seen_common`: the same loss on the |signed log y| band below 1, and on concepts with 50 or more training filings.
- `d_pred_shuffle`: median absolute change in prediction when the probe filings' values are shuffled. It shows whether the model reads context.
- W only:
  - `P_mean`, `P_min`: drawn subsets per target over the window.
  - `usable_frac`: the share of drawn subsets that don't contain their target.
  - `zero_frac`: the share of targets with no usable subset. It is exact over every microbatch in the window.
- `lr`: the learning rate at that point.
- `train`, `eval`: cumulative hours.

Printed at the end, `epochs`: per arm and epoch:

- `heldout`, the bands and groups, `d_pred_shuffle`
- `opt_steps`
- `train_h` and `eval_h` for that epoch alone
- `peak_gib`
- `targets_per_step` (median, 90th percentile, maximum)

Files under `E:\apriori-ml-v3\pilot\runs`:

- `v3_{arm}_last.pt`
- `v3_{arm}_epoch{1,2}.pt`
- `v3_{arm}_curve.pkl.lz4`

In [5]:
# Estimated run time: about 5.6 hours on the 2080 from initialization for the three arms (W0_registered 1.1 h, W0 0.7 h, W 3.8 h, 
#  each including 21 evaluations);
# about 2 minutes when the three checkpoints are complete (each arm loads its checkpoint under its contract and skips).
# Resumable: rerun the cell after any interruption. Runs after cell 5, same kernel.
ARMS_TO_RUN = ["W0_registered", "W0", "W"]
assert set(ARMS_TO_RUN) <= set(ARM_SPEC)

def arm_paths(arm):
    return os.path.join(RUN_DIR, f"v3_{arm}_last.pt"), os.path.join(RUN_DIR, f"v3_{arm}_curve.pkl.lz4")

def save_state(path, curve_path, model, opt, st, curve, contract):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "st": st, "curve": curve, "contract": contract,
                "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all()}, path + ".tmp")
    os.replace(path + ".tmp", path)
    save_to_pickle(curve, curve_path, lz4_compression=True)          # atomic: mkstemp in the same folder, then os.replace

def window_stats(st):
    if st["Wt"] == 0:
        return {}
    return {"P_mean": st["Psum"] / st["Wt"], "P_min": st["Pmin"], "usable_frac": st["U"] / max(st["Psum"], 1),
            "zero_frac": st["Z"] / st["Wt"], "window_targets": st["Wt"]}

def run_arm(arm):
    sp = ARM_SPEC[arm]
    kind, rule, walk, ranked = sp["kind"], sp["rule"], sp["walk"], sp["ranked"]
    contract = arm_contract(arm)
    last, curve_path = arm_paths(arm)
    ds = mpd.PilotDataset(indptr, indices, data, pdat["file_names"], ctx12, kind, M=M_W, run_factor=RUN_FACTOR, seed=SEED,
                          rate=MASK_RATE, structures=sp["structures"])
    model = new_model()
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        assert ck["contract"] == contract, f"{last} was written under contract {ck['contract']}, not {contract}"
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st, curve = ck["st"], ck["curve"]
        torch.set_rng_state(ck["rng_cpu"]); torch.cuda.set_rng_state_all(ck["rng_cuda"])
        print(f"[{arm}] resumed: epoch index {st['e']}, microbatch {st['k']:,}, steps {st['steps']:,}")
    else:
        assert not os.path.exists(curve_path), f"{curve_path} exists without a checkpoint; delete it to start {arm} over"
        _, ev0, t_ev = evaluate(model, ranked)
        st = {"e": 0, "k": 0, "g": 0, "t": 0, "steps": 0, "train_s": 0.0, "eval_s": t_ev, "gwin": [], "gmax": 0.0, "uwin": [], "ups": [],
              "tps": [], "peak": 0.0, "stalls": [], "pending_ckpt": False, "Psum": 0, "Pmin": np.inf, "U": 0, "Z": 0, "Wt": 0, "Swin": [],
              "train_s_at_epoch": {0: 0.0}, "eval_s_at_epoch": {0: t_ev}}
        curve = {"contract": contract, "arm": arm, "params": arm_params(arm), "sources_sha256": SOURCES_SHA256, "data_sha256": DATA_SHA256,
                 "environment": ENVIRONMENT, "y_ho_sha16": sha16(y_ho.tobytes()), "rows": [], "epochs": {}}
        curve["rows"].append({"arm": arm, "epoch": 0, "pct": 0, "opt_steps": 0, "lr": 0.0, **ev0, "train_h": 0.0, "eval_h": t_ev / 3600})
        save_state(last, curve_path, model, opt, st, curve, contract)
        print(f"[{arm}] init: heldout {ev0['heldout']:.4f} | evaluation {t_ev:.0f} s")
    if all(ep in curve["epochs"] for ep in range(1, EPOCHS_RUN + 1)):
        print(f"[{arm}] complete; skipping")
        del model, opt, params, ds
        torch.cuda.empty_cache()
        return curve

    model.train()
    opt.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    acc_n = 0

    def do_step(count, lr):
        if rule == "D4" or count > 1:
            for p in params:
                if p.grad is not None:
                    p.grad.div_(count)
        grads = [p.grad for p in params if p.grad is not None]
        if not grads or not all(bool(torch.isfinite(q).all()) for q in grads):
            raise RuntimeError(f"[{arm}] missing or nonfinite gradient at step {st['steps']}; stopped before the update")
        st["gmax"] = max(st["gmax"], float(torch.sqrt(sum((q.double() ** 2).sum() for q in grads))))
        for gr in opt.param_groups:
            gr["lr"] = lr
        before = [p.detach().clone() for p in params]
        opt.step()
        st["ups"].append(float(torch.sqrt(sum(((p.detach() - b).double() ** 2).sum() for p, b in zip(params, before)))))
        opt.zero_grad(set_to_none=True)
        st["steps"] += 1; st["tps"].append(count)

    for e in range(st["e"], EPOCHS_RUN):
        if (e + 1) in curve["epochs"]:
            continue
        keys = KEYS[e]
        epoch_done = False
        while not epoch_done:
            gen = torch.Generator()                                    # dedicated: worker seeds never draw on the global RNG
            gen.manual_seed(int(sha16(f"loader-{SEED}-{arm}-{e}-{st['k']}".encode()), 16) % 2**63)
            loader = DataLoader(ds, batch_sampler=keys[st["k"]:], collate_fn=mpd.collate_list, num_workers=WORKERS,
                                prefetch_factor=2, timeout=LOADER_TIMEOUT, generator=gen)
            t_seg = time.perf_counter()
            try:
                for items in loader:
                    k = st["k"]
                    assert [(it["i"], tuple(int(t) for t in it["tgt"])) for it in items] == [(key[0], tuple(key[3])) for key in keys[k]], \
                        f"[{arm}] loader out of plan order at microbatch {k}"
                    n = sum(len(it["tgt"]) for it in items)
                    loss, expo, _, _ = mcb.batch_loss(model, kind, items, NT, dev, walk=walk)
                    if not bool(torch.isfinite(loss)):
                        raise RuntimeError(f"[{arm}] nonfinite loss at epoch {e + 1} microbatch {k}")
                    if kind == "W":
                        ps = [int(it["P"]) for it in items for _ in it["tgt"]]
                        st["Psum"] += sum(ps); st["Pmin"] = min(st["Pmin"], min(ps)); st["U"] += int(expo) - n
                        st["Z"] += sum(tr is None for it in items for tr in it["tries_excl"]); st["Wt"] += n
                    st["k"] += 1; st["t"] += n
                    end = st["k"] == MICROBATCHES_PER_EPOCH
                    if rule == "registered":
                        loss.backward()
                        do_step(1, lr_registered(st["g"])); st["g"] += 1
                    else:
                        (loss * n).backward()
                        acc_n += n
                        if acc_n >= MIN_TARGETS or end:
                            do_step(acc_n, lr_d4(st["t"])); acc_n = 0
                            if st["pending_ckpt"]:
                                st["train_s"] += time.perf_counter() - t_seg
                                st["pending_ckpt"] = False
                                save_state(last, curve_path, model, opt, st, curve, contract)
                                t_seg = time.perf_counter()
                    if st["k"] in BOUNDS:
                        st["train_s"] += time.perf_counter() - t_seg
                        st["peak"] = max(st["peak"], torch.cuda.max_memory_allocated() / 2**30)
                        ups = np.array(st["ups"]) if st["ups"] else np.array([np.nan])
                        st["uwin"].append(round(float(np.median(ups)), 4)); st["ups"] = []
                        st["gwin"].append(round(st["gmax"], 2)); st["gmax"] = 0.0
                        ws = window_stats(st)
                        if ws:
                            st["Swin"].append(ws)
                        st.update({"Psum": 0, "Pmin": np.inf, "U": 0, "Z": 0, "Wt": 0})
                        pct = BOUNDS.index(st["k"]) * 10 + 10
                        l_new, ev, t_ev = evaluate(model, ranked)
                        torch.cuda.reset_peak_memory_stats()
                        st["eval_s"] += t_ev
                        lr_now = lr_registered(st["g"]) if rule == "registered" else lr_d4(st["t"])
                        row = {"arm": arm, "epoch": e + 1, "pct": pct, "opt_steps": st["steps"], "lr": lr_now, **ev, **ws,
                               "train_h": st["train_s"] / 3600, "eval_h": st["eval_s"] / 3600}
                        curve["rows"].append(row)
                        wtxt = (f" | P_mean {ws['P_mean']:.0f} | P_min {ws['P_min']:.0f} | usable_frac {ws['usable_frac']:.3f} | "
                                f"zero_frac {ws['zero_frac']:.4f}") if ws else ""
                        print(f"[{arm}] epoch {e + 1} {pct}%: steps {st['steps']:,} | heldout {ev['heldout']:.4f} | 0 to 1 {ev['0 to 1']:.4f} | "
                              f"seen_common {ev['seen_common']:.4f} | d_pred_shuffle {ev['d_pred_shuffle']:.4g}{wtxt} | lr {lr_now:.2e} | "
                              f"train {row['train_h']:.2f} h | eval {row['eval_h']:.2f} h")
                        if end:
                            want_steps = STEPS_AT_EPOCH[e + 1] if rule == "D4" else (e + 1) * MICROBATCHES_PER_EPOCH
                            assert st["steps"] == want_steps, f"[{arm}] {st['steps']:,} steps at epoch {e + 1}, expected {want_steps:,}"
                            tps = np.array(st["tps"])
                            st["train_s_at_epoch"][e + 1], st["eval_s_at_epoch"][e + 1] = st["train_s"], st["eval_s"]
                            curve["epochs"][e + 1] = {"losses": l_new, "row": row, "grad_max_windows": st["gwin"], "upd_med_windows": st["uwin"],
                                                      "sampler_windows": st["Swin"], "peak_gib": st["peak"],
                                                      "train_h": (st["train_s"] - st["train_s_at_epoch"][e]) / 3600,
                                                      "eval_h": (st["eval_s"] - st["eval_s_at_epoch"][e]) / 3600,
                                                      "targets_per_step": (float(np.median(tps)), float(np.percentile(tps, 90)), int(tps.max()))}
                            st.update({"e": e + 1, "k": 0, "gwin": [], "uwin": [], "tps": [], "Swin": [], "peak": 0.0, "pending_ckpt": False})
                            save_state(os.path.join(RUN_DIR, f"v3_{arm}_epoch{e + 1}.pt"), curve_path, model, opt, st, curve, contract)
                            save_state(last, curve_path, model, opt, st, curve, contract)
                            epoch_done = True
                        elif rule == "registered" or acc_n == 0:
                            save_state(last, curve_path, model, opt, st, curve, contract)
                        else:
                            st["pending_ckpt"] = True
                        t_seg = time.perf_counter()
                        if epoch_done:
                            break
            except RuntimeError as ex:
                if "DataLoader timed out" not in str(ex):
                    raise
                st["stalls"].append((e, st["k"]))
                print(f"[{arm}] loader stalled at epoch {e + 1} microbatch {st['k']:,}; rebuilding (stall {len(st['stalls'])} of at most {MAX_STALLS})")
                if len(st["stalls"]) >= MAX_STALLS:
                    raise RuntimeError(f"[{arm}] {MAX_STALLS} loader stalls; stopping") from None
                st["train_s"] += time.perf_counter() - t_seg
            finally:
                del loader
    del model, opt, params, ds
    torch.cuda.empty_cache()
    return curve

curves = {arm: run_arm(arm) for arm in ARMS_TO_RUN}

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
erows = []
for arm, cv in curves.items():
    for ep in sorted(cv["epochs"]):
        info, r = cv["epochs"][ep], cv["epochs"][ep]["row"]
        erows.append({"arm": arm, "epoch": ep, **{k: r[k] for k in ("heldout", "0 to 1", "1 to 3", "3 to 10", "seen_common", "seen_rare", "unseen",
                                                                   "d_pred_shuffle", "opt_steps")},
                      "train_h": info["train_h"], "eval_h": info["eval_h"], "peak_gib": info["peak_gib"],
                      "targets_per_step": info["targets_per_step"]})
epochs_table = pd.DataFrame(erows)
print("epochs"); print(epochs_table.to_string(index=False))
pd.reset_option("display.float_format")
epochs_table

[W0_registered] init: heldout 0.2362 | evaluation 2 s
[W0_registered] epoch 1 10%: steps 12,677 | heldout 0.2251 | 0 to 1 0.0739 | seen_common 0.2220 | d_pred_shuffle 0.0008213 | lr 3.00e-04 | train 0.07 h | eval 0.00 h
[W0_registered] epoch 1 20%: steps 25,353 | heldout 0.2228 | 0 to 1 0.0735 | seen_common 0.2197 | d_pred_shuffle 1.376e-05 | lr 3.00e-04 | train 0.12 h | eval 0.00 h
[W0_registered] epoch 1 30%: steps 38,030 | heldout 0.2226 | 0 to 1 0.0729 | seen_common 0.2192 | d_pred_shuffle 1.796e-06 | lr 3.00e-04 | train 0.18 h | eval 0.00 h
[W0_registered] epoch 1 40%: steps 50,707 | heldout 0.2226 | 0 to 1 0.0739 | seen_common 0.2189 | d_pred_shuffle 2.259e-05 | lr 2.99e-04 | train 0.23 h | eval 0.00 h
[W0_registered] epoch 1 50%: steps 63,384 | heldout 0.2224 | 0 to 1 0.0762 | seen_common 0.2185 | d_pred_shuffle 7.153e-07 | lr 2.98e-04 | train 0.29 h | eval 0.00 h
[W0_registered] epoch 1 60%: steps 76,060 | heldout 0.2213 | 0 to 1 0.0721 | seen_common 0.2181 | d_pred_shuffle 2.2

,arm,epoch,heldout,0 to 1,1 to 3,3 to 10,seen_common,seen_rare,unseen,d_pred_shuffle,opt_steps,train_h,eval_h,peak_gib,targets_per_step
0,W0_registered,1,0.220623,0.072607,1.070091,3.170666,0.216903,0.224135,0.267844,0.000001,126767,0.559351,0.006195,0.496605,"(1.0, 1.0, 1)"
1,W0_registered,2,0.218783,0.072695,1.054264,3.155370,0.214999,0.222344,0.266829,0.000511,253534,0.562993,0.006245,0.496605,"(1.0, 1.0, 1)"
2,W0,1,0.184906,0.072423,0.804282,2.650196,0.176595,0.208116,0.253042,0.194763,32995,0.346206,0.006169,0.502744,"(35.0, 48.0, 63)"
3,W0,2,0.166509,0.054511,0.785673,2.600144,0.158277,0.190843,0.230733,0.129266,65953,0.345931,0.006164,0.502744,"(35.0, 48.0, 63)"
4,W,1,0.175292,0.061326,0.810481,2.607808,0.167243,0.197886,0.240995,0.132831,32995,1.884363,0.006486,0.829011,"(35.0, 48.0, 63)"
5,W,2,0.158574,0.053012,0.738538,2.483233,0.149835,0.186084,0.222666,0.141962,65953,1.887564,0.006511,0.819109,"(35.0, 48.0, 63)"


## Cell 7. Paired intervals and bin results

Reads the saved curves; no training. All comparisons are per held out target (136,446 targets in 998 filings). Every loss array is checked to have one finite value per held out target before it is compared or bootstrapped. Intervals are 95%, from a bootstrap over filings (2,000 resamples, seed (367, 91)). They measure held out filing sampling uncertainty for these trained models. They do not include training seed variability. A verdict is given only when both interval bounds are finite. All three arms use full context evaluation.

Printed labels, in order:

- `endpoints`: per comparison and epoch, `a` and `b` (the pooled means), `d_mean` (a minus b; negative favors a), `ci_lo`, `ci_hi` and `verdict` ("a better", "b better" or "no difference established"). Comparisons:
  - `W minus W0`: the registered primary endpoint at epoch 2. The epoch 1 row is descriptive.
  - `W0 minus W0_registered`: the D4 against registered comparison.
- `tradeoffs`: for `W minus W0`, per epoch, by decile bin, magnitude band and concept group. Groups with no targets are skipped (the `10+` band is empty on cleaned data). Columns: `targets`, `a`, `b`, `d_mean`, `ci_lo`, `ci_hi` and `reading`. These are reported, not gating. The intervals are nominal, and epoch 1 is descriptive.
- `saved`: the report's path, written to `E:\apriori-ml-v3\pilot\runs\v3_pilot_report.pkl.lz4` through `save_to_pickle` (a temporary file in the same folder, then `os.replace`), with both tables and the curves' contracts.curves' contracts.

In [6]:
# Estimated run time: under 2 minutes (loads three curves, about 50 filing bootstraps of 2,000 resamples each). Runs after
# cell 6, same kernel; reads the saved curves, no training.
pd.set_option("display.float_format", lambda v: f"{v:.6g}")

def checked(arr, name):
    a = np.asarray(arr, dtype=np.float64)
    assert a.shape == y_ho.shape, f"{name}: shape {a.shape}, expected {y_ho.shape}"
    assert np.isfinite(a).all(), f"{name}: nonfinite loss"
    return a

cv3 = {arm: load_from_pickle(arm_paths(arm)[1]) for arm in ARM_SPEC}
for arm, cv in cv3.items():
    assert cv["contract"] == arm_contract(arm), f"{arm} curve contract differs"
    assert all(ep in cv["epochs"] for ep in (1, 2)), f"{arm} is missing an epoch"
    assert cv["y_ho_sha16"] == sha16(y_ho.tobytes()), f"{arm} curve was scored on a different held out set"
LOSS = {(arm, ep): checked(cv3[arm]["epochs"][ep]["losses"], f"{arm} epoch {ep}") for arm in ARM_SPEC for ep in (1, 2)}

def verdict(lo, hi):
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "not estimable"
    return "a better" if hi < 0 else ("b better" if lo > 0 else "no difference established")

# ----- endpoints -----
COMPARE = [("W minus W0",             lambda ep: LOSS[("W", ep)],  lambda ep: LOSS[("W0", ep)]),
           ("W0 minus W0_registered", lambda ep: LOSS[("W0", ep)], lambda ep: LOSS[("W0_registered", ep)])]
erow = []
for name, fa, fb in COMPARE:
    for ep in (1, 2):
        a, b = fa(ep), fb(ep)
        m_, lo, hi = cluster_ci(a - b, fid)
        erow.append({"comparison": name, "epoch": ep, "a": float(a.mean()), "b": float(b.mean()), "d_mean": m_, "ci_lo": lo, "ci_hi": hi,
                     "verdict": verdict(lo, hi)})
endpoints = pd.DataFrame(erow)
print("endpoints"); print(endpoints.to_string(index=False))

# ----- tradeoffs -----
GROUPS = [(k, m) for k, m in list(bins.items()) + list(bands.items()) + list(groups.items()) if m.any()]   # empty groups skipped
trow = []
for name, fa, fb in COMPARE[:1]:
    for ep in (1, 2):
        a, b = fa(ep), fb(ep)
        d = a - b
        for gname, mk in GROUPS:
            m_, lo, hi = cluster_ci(d[mk], fid[mk])
            trow.append({"comparison": name, "epoch": ep, "group": gname, "targets": int(mk.sum()), "a": float(a[mk].mean()),
                         "b": float(b[mk].mean()), "d_mean": m_, "ci_lo": lo, "ci_hi": hi, "reading": verdict(lo, hi)})
tradeoffs = pd.DataFrame(trow)
print("tradeoffs"); print(tradeoffs.to_string(index=False))

REPORT = os.path.join(RUN_DIR, "v3_pilot_report.pkl.lz4")
save_to_pickle({"written": datetime.datetime.now().isoformat(timespec="seconds"),          # atomic: mkstemp in the same folder, then os.replace
                "contracts": {arm: cv["contract"] for arm, cv in cv3.items()}, "sources_sha256": SOURCES_SHA256,
                "boot": BOOT, "boot_seed": [SEED, BOOT_TAG],
                "tables": {"endpoints": endpoints, "tradeoffs": tradeoffs}},
               REPORT, lz4_compression=True)
print("saved"); print(pd.DataFrame([{"path": REPORT, "kib": os.path.getsize(REPORT) / 1024, "sha16": sha256_file(REPORT)[:16]}]).to_string(index=False))
pd.reset_option("display.float_format")

endpoints
            comparison  epoch        a        b      d_mean       ci_lo       ci_hi  verdict
            W minus W0      1 0.175292 0.184906 -0.00961391  -0.0109838  -0.0081744 a better
            W minus W0      2 0.158574 0.166509 -0.00793564 -0.00907771 -0.00689331 a better
W0 minus W0_registered      1 0.184906 0.220623   -0.035717  -0.0399223  -0.0320387 a better
W0 minus W0_registered      2 0.166509 0.218783  -0.0522734  -0.0563864  -0.0485955 a better
tradeoffs
comparison  epoch       group  targets          a          b      d_mean        ci_lo        ci_hi                   reading
W minus W0      1       bin 0    15290 0.00773491  0.0143843 -0.00664939  -0.00771028  -0.00557411                  a better
W minus W0      1       bin 1    16274  0.0103903  0.0181088 -0.00771853  -0.00888852   -0.0065527                  a better
W minus W0      1       bin 2    15752  0.0152982  0.0260474  -0.0107492    -0.012227  -0.00924059                  a better
W minus W0     

## Cell 8. Matched cost study: shared cache, reference walk and direct enumeration

On the same sampled microbatches, the W training objective runs forward and backward in the D4 form (`loss × n`) three ways:

- **cache**: `batch_loss` with `walk="cache"`, the shared cache.
- **reference walk**: `batch_loss` with pointer jumping, implementation (c), the pilot's own traversal.
- **direct**: chunked direct enumeration. Each subset gets one masked attention, scoped to the target's filing. Gradients accumulate in row chunks into per target leaves, followed by one backward through the encoder.

There is no loader, optimizer, evaluation or per step diagnostics. The model is the certified initialization in train mode for all three paths. It has no dropout (asserted), so the paths are deterministic and need no RNG matching. Settings:

- 6 microbatches per size class (single 1 target, small 2 to 8, medium 9 to 31, full 32), drawn from the epoch 1 plan with seed (367, 223). The picks and the class counts are printed and saved.
- Direct chunks of 1,024 rows, with a sweep over 256, 1,024 and 4,096 rows for the full class.
- Equivalence per parameter tensor: absolute 1e-6 plus relative 1e-4 of the tensor's largest gradient, and loss relative difference below 1e-4.

**What this measures.** Execution of three implementations of one objective on the same inputs. The direct path is a scoped, target by target baseline. A direct implementation that batched targets could be faster, so the paper's general systems claims stay with Section 7.

**Certification before timing.** On the class's first microbatch, the reference walk and every direct chunk size the projection uses must reproduce the cache path's loss and every parameter gradient. The smallest chunk is tried first. A chunk that runs out of memory is recorded as OOM and never timed. Certification gradients are copied to CPU and released before any timing, so no diagnostic storage is counted in the memory peaks.

**Out of memory during timing.** A direct chunk can fit the first microbatch and still fail on a later one. The class is then timed at the next smaller certified chunk. If no certified chunk runs all six microbatches, the class row says so, and the direct projection is reported as unavailable. Ratios and totals are computed only from finite times.

Printed labels, in order:

- `picked microbatches`: per class, `epoch_count` (microbatches of that size class in epoch 1) and `picked` (the six plan indices timed).
- `EQUIVALENCE`: per class and path (`reference walk`, or `direct` with its `chunk_rows`):
  - `status` (ok or OOM), `rel_loss_diff`, `worst_param`, `worst_err_over_tol` (below 1 passes)
  - `subsets`, `usable`: subsets in the class's first microbatch, and how many are usable.
- `EXECUTION COST BY CLASS`: per class, mean seconds per microbatch:
  - `cache_exec_s`, `ref_exec_s`: forward and backward through `batch_loss`, each including its own serial plan build.
  - `direct_exec_s` at `direct_chunk`, the chunk that ran all six microbatches, and `direct_timing` (ok, or OOM in timing at every certified chunk).
  - `direct_prep_s`: direct's serial CPU membership and weights.
  - `cache_struct_s`, `ref_struct_s`: serial dataset construction of each path's subset structures, measured without workers. The cache's structures are built from the subset list by the canonical builder. The pilot's families are built per target without that target, so the run trie generator of Section 7 does not apply here.
  - `cache_gib`, `ref_gib`, `direct_gib`: peak allocated memory during each path's timing.
  - `ref_over_cache`, `direct_over_cache`: execution ratios. Above 1 means the cache is faster.
- `CHUNK SIZE SWEEP (full class)`: `direct_exec_s`, `direct_gib` and `status` (ok, OOM at certification, or OOM in timing) per chunk size.
- `HOURS PER EPOCH`: the class means times the epoch's class counts.
  - `cache_exec_h`, `ref_exec_h`, `direct_exec_h` (the full class at its best chunk that ran all six microbatches), `direct_best_chunk`.
  - `cache_struct_h`, `ref_struct_h`, `direct_prep_h`.
  - `ref_over_cache` and `direct_over_cache` (execution).
  - `cache_total_h` (execution plus structures), `ref_total_h` (execution plus structures) and `direct_total_h` (execution plus direct preparation plus the reference trie construction). The trie construction is counted because this direct implementation reads its subsets from the reference tries. It is not a preparation step that direct attention requires in general.
  - `direct_projection`: ok, or unavailable when any class lacks a finite direct time.
  - These are forward and backward hours, not end to end training hours.
- `saved`: `E:\apriori-ml-v3\pilot\runs\v3_cost_study.pkl.lz4`, written through `save_to_pickle` (a temporary file in the same folder, then `os.replace`).file in the same folder, then `os.replace`).

In [7]:
# Estimated run time: 3 to 8 minutes on the 2080 (one arm x four classes x six microbatches, three paths each, direct
# preparation on CPU, the full class chunk sweep). Runs after cell 7, same kernel; no training.
PER_CLASS, CHUNK_ROWS, CHUNK_SWEEP, ATOL, RTOL = 6, 1024, (256, 1024, 4096), 1e-6, 1e-4
CLASSES = {"single": (1, 1), "small": (2, 8), "medium": (9, 31), "full": (32, 32)}

keys0 = KEYS[0]
sizes = np.array([len(k) for k in keys0])
rng = np.random.default_rng((SEED, 223))
picked = {cls: sorted(rng.choice(np.flatnonzero((sizes >= lo) & (sizes <= hi)), PER_CLASS, replace=False).tolist()) for cls, (lo, hi) in CLASSES.items()}
counts = {cls: int(((sizes >= lo) & (sizes <= hi)).sum()) for cls, (lo, hi) in CLASSES.items()}
print("picked microbatches"); print(pd.DataFrame([{"class": c, "epoch_count": counts[c], "picked": picked[c]} for c in CLASSES]).to_string(index=False))

def build_prep(kind, items):
    """Direct baseline preparation. Serial CPU: per target, subset membership over its own scope, usable flags and fixed loss weights matching mc_target_loss."""
    prep = []
    for b, it in enumerate(items):
        n_b = int(it["n"])
        for j, t_local in enumerate(np.asarray(it["tgt"], dtype=np.int64)):
            tr, scope, inv = it["trie"], np.arange(n_b), np.arange(n_b)
            full = np.ones(n_b, dtype=bool); full[t_local] = False
            pos, parent, term = (np.asarray(tr[k_], dtype=np.int64) for k_ in ("pos", "parent", "terminals"))
            assert (inv[pos] >= 0).all(), "tree position outside the target's scope"
            P = len(term)
            member = np.zeros((P, len(scope)), dtype=bool)
            rows, cur = np.arange(P), term.copy()
            while (cur >= 0).any():
                a = cur >= 0
                member[rows[a], inv[pos[cur[a]]]] = True
                cur = np.where(a, parent[np.maximum(cur, 0)], -1)
            k = member.sum(1)
            usable = (~member[:, inv[t_local]]) & (k > 0)
            w = np.where(usable, 0.5 / mcm.w_subset_denominator(n_b, int(it["P"])), 0.0)
            w_full = 0.5
            prep.append({"b": b, "scope": scope, "full": full, "member": member, "rows_use": np.flatnonzero(usable), "usable": usable, "w": w, "w_full": w_full})
    return prep

def direct_step(model, kind, items, prep, chunk_rows, check):
    """Direct enumeration forward and backward; leaves the gradient of (loss x n) in the parameters."""
    ns = np.array([it["n"] for it in items], dtype=np.int64)
    offs = np.concatenate([[0], np.cumsum(ns)[:-1]])
    cols = torch.as_tensor(np.concatenate([it["cols"] for it in items]), device=dev)
    vals = torch.as_tensor(np.concatenate([it["vals"] for it in items]), device=dev).to(NT.dtype)
    nv = NT[cols]
    ch_item = np.concatenate([np.full(len(it["tgt"]), b) for b, it in enumerate(items)])
    ch_global = offs[ch_item] + np.concatenate([np.asarray(it["tgt"], dtype=np.int64) for it in items])
    C = len(ch_global)
    if check:
        assert C == len(items) == len(prep), "one target per item is assumed for the D4 form"
    tg = torch.as_tensor(ch_global, device=dev)
    z, V, e = model.scores(nv, vals, nv[tg])
    y = signed_log(vals)[tg]
    zd, Vd, ed = z.detach(), V.detach(), e.detach()
    gz, gV, ge = torch.zeros_like(zd), torch.zeros_like(Vd), torch.zeros_like(ed)
    total = torch.zeros((), dtype=zd.dtype, device=dev)
    zero1 = torch.zeros(1, dtype=torch.long, device=dev)
    for c, p in enumerate(prep):
        sc = torch.as_tensor(offs[p["b"]] + p["scope"], device=dev)
        zc = zd[sc, c][:, None, :].clone().requires_grad_(True)
        Vc = Vd[sc].clone().requires_grad_(True)
        ec = ed[c][None].clone().requires_grad_(True)
        fp, fL, fk, fu = mcm.mc_masked_attention(zc, Vc, torch.as_tensor(p["full"], device=dev)[None], zero1)
        if check:
            assert bool(fu.all()), "empty full context"
        lf = p["w_full"] * F.huber_loss(model.predict(fp, fL, fk, ec), y[c][None], delta=1.0, reduction="none").sum()
        lf.backward()
        total += lf.detach()
        rows_use = p["rows_use"]
        if len(rows_use):
            mem = torch.as_tensor(p["member"], device=dev)
            w = torch.as_tensor(p["w"], device=dev, dtype=zd.dtype)
            for s in range(0, len(rows_use), chunk_rows):
                idx = torch.as_tensor(rows_use[s:s + chunk_rows], device=dev)
                sp, sL, sk, su = mcm.mc_masked_attention(zc, Vc, mem[idx], torch.zeros(len(idx), dtype=torch.long, device=dev))
                if check:
                    assert bool(su.all())
                l = F.huber_loss(model.predict(sp, sL, sk, ec.expand(len(idx), -1)), y[c].expand(len(idx)), delta=1.0, reduction="none")
                lw = (w[idx] * l).sum()
                lw.backward()
                total += lw.detach()
        gz[sc, c] += zc.grad[:, 0, :]; gV[sc] += Vc.grad; ge[c] += ec.grad[0]
    torch.autograd.backward((z, V, e), (gz, gV, ge))
    return float(total) / C

def named_grads(model):
    """Certification copies, on CPU, so no diagnostic storage stays on the GPU."""
    return [(n, p.grad.detach().double().cpu() if p.grad is not None else None) for n, p in model.named_parameters()]

def path_fb(model, kind, items, walk):
    assert all(len(it["tgt"]) == 1 for it in items), "column schedule items carry one target"
    loss = mcb.batch_loss(model, kind, items, NT, dev, walk=walk)[0]
    (loss * len(items)).backward()
    return float(loss)

def certify(g_ref, loss_ref, g_other, loss_other, label):
    """Per tensor absolute ATOL plus RTOL x the tensor's largest reference gradient; loss relative below RTOL."""
    assert len(g_ref) == len(g_other) and math.isfinite(loss_other), f"{label}: gradient lists differ or nonfinite loss"
    worst, worst_name = 0.0, ""
    for (n_, a), (m_, b) in zip(g_ref, g_other):
        assert n_ == m_ and (a is None) == (b is None), f"{label}: different parameters received gradients"
        if a is None:
            continue
        assert torch.isfinite(a).all() and torch.isfinite(b).all(), f"{label}: nonfinite gradient in {n_}"
        r = float(((a - b).abs() / (ATOL + RTOL * a.abs().max())).max())
        if r > worst:
            worst, worst_name = r, n_
    ldiff = abs(loss_ref - loss_other) / max(abs(loss_ref), 1e-12)
    assert ldiff < RTOL and worst <= 1.0, f"{label}: differs from the cache path (loss rel {ldiff:.3g}; {worst_name} err/tol {worst:.3g}); no timing taken"
    return ldiff, worst, worst_name

def time_path(model, kind, batches, walk):
    for items in batches[:2]:
        path_fb(model, kind, items, walk); model.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
    for items in batches:
        path_fb(model, kind, items, walk); model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    s, gib = (time.perf_counter() - t0) / len(batches), torch.cuda.max_memory_allocated() / 2**30
    assert math.isfinite(s), f"{kind} {walk}: nonfinite time"
    return s, gib

def time_direct(model, kind, batches, preps, chunk):
    """Mean seconds and peak GiB over all six microbatches, or (nan, nan) if any of them runs out of memory."""
    try:
        direct_step(model, kind, batches[0], preps[0], chunk, check=False); model.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
        for items, pr in zip(batches, preps):
            direct_step(model, kind, items, pr, chunk, check=False); model.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / len(batches), torch.cuda.max_memory_allocated() / 2**30
    except torch.cuda.OutOfMemoryError:
        model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
        return float("nan"), float("nan")

def struct_s(ds, ks):
    t0 = time.perf_counter()
    for i in ks:
        [ds[key] for key in keys0[i]]
    return (time.perf_counter() - t0) / len(ks)

rows_eq, rows_t, sweep = [], [], []
for kind in ("W",):
    ds_exec = ds_W["both"]
    ds_struct = {"cache": ds_W["cache"], "ref": ds_W["v1"]}
    for cls, ks in picked.items():
        batches = [[ds_exec[key] for key in keys0[i]] for i in ks]
        cache_struct, ref_struct = struct_s(ds_struct["cache"], ks), struct_s(ds_struct["ref"], ks)
        model = new_model()
        assert model.training and not any(isinstance(m, torch.nn.Dropout) for m in model.modules()), "model mode or dropout differs from the declared protocol"
        # ----- certification on the first microbatch, against the cache path; copies on CPU, released after each check -----
        model.zero_grad(set_to_none=True); loss_c = path_fb(model, kind, batches[0], "cache"); g_c = named_grads(model); model.zero_grad(set_to_none=True)
        assert math.isfinite(loss_c), f"{kind} {cls}: nonfinite cache loss"
        prep0 = build_prep(kind, batches[0])
        n_sub, n_use = int(sum(len(p["usable"]) for p in prep0)), int(sum(p["usable"].sum() for p in prep0))
        loss_v = path_fb(model, kind, batches[0], REF_WALK[kind]); g_v = named_grads(model); model.zero_grad(set_to_none=True)
        ld, wo, wn = certify(g_c, loss_c, g_v, loss_v, f"{kind} {cls} reference walk")
        del g_v
        rows_eq.append({"arm": kind, "class": cls, "path": "reference walk", "chunk_rows": np.nan, "status": "ok", "rel_loss_diff": ld,
                        "worst_param": wn, "worst_err_over_tol": wo, "subsets": n_sub, "usable": n_use})
        chunks_needed = sorted(set(CHUNK_SWEEP) | {CHUNK_ROWS}) if cls == "full" else sorted({min(CHUNK_SWEEP), CHUNK_ROWS})
        certified = {}
        for chunk in chunks_needed:
            try:
                loss_d = direct_step(model, kind, batches[0], prep0, chunk, check=True)
            except torch.cuda.OutOfMemoryError:
                model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                certified[chunk] = False
                rows_eq.append({"arm": kind, "class": cls, "path": "direct", "chunk_rows": chunk, "status": "OOM", "rel_loss_diff": np.nan,
                                "worst_param": "", "worst_err_over_tol": np.nan, "subsets": n_sub, "usable": n_use})
                continue
            g_d = named_grads(model); model.zero_grad(set_to_none=True)
            ld, wo, wn = certify(g_c, loss_c, g_d, loss_d, f"{kind} {cls} direct chunk {chunk}")
            del g_d
            certified[chunk] = True
            rows_eq.append({"arm": kind, "class": cls, "path": "direct", "chunk_rows": chunk, "status": "ok", "rel_loss_diff": ld,
                            "worst_param": wn, "worst_err_over_tol": wo, "subsets": n_sub, "usable": n_use})
        del g_c
        torch.cuda.empty_cache()
        assert any(certified.values()), f"{kind} {cls}: no direct chunk size fits on this card"
        # ----- execution: cache, reference walk, direct -----
        cache_s, cache_gib = time_path(model, kind, batches, "cache")
        ref_s, ref_gib = time_path(model, kind, batches, REF_WALK[kind])
        t0 = time.perf_counter()
        preps = [build_prep(kind, items) for items in batches]
        prep_s = (time.perf_counter() - t0) / len(batches)
        ok_chunks = [c_ for c_, ok_ in certified.items() if ok_]
        candidates = sorted([c_ for c_ in ok_chunks if c_ <= CHUNK_ROWS], reverse=True) + sorted(c_ for c_ in ok_chunks if c_ > CHUNK_ROWS)
        dir_s, dir_gib, used_chunk, timed = float("nan"), float("nan"), None, {}
        for chunk in candidates:                                        # 1,024 first, then smaller certified chunks
            timed[chunk] = time_direct(model, kind, batches, preps, chunk)
            if math.isfinite(timed[chunk][0]):
                dir_s, dir_gib = timed[chunk]; used_chunk = chunk
                break
        direct_timing = "ok" if used_chunk is not None else "OOM in timing at every certified chunk"
        if cls == "full":
            for chunk in CHUNK_SWEEP:
                if not certified.get(chunk):
                    sweep.append({"arm": kind, "chunk_rows": chunk, "direct_exec_s": np.nan, "direct_gib": np.nan, "status": "OOM at certification"})
                    continue
                s_, g_ = timed[chunk] if chunk in timed else time_direct(model, kind, batches, preps, chunk)
                sweep.append({"arm": kind, "chunk_rows": chunk, "direct_exec_s": s_, "direct_gib": g_, "status": "ok" if math.isfinite(s_) else "OOM in timing"})
        rows_t.append({"arm": kind, "class": cls, "direct_chunk": used_chunk, "direct_timing": direct_timing,
                       "cache_exec_s": cache_s, "ref_exec_s": ref_s, "direct_exec_s": dir_s,
                       "direct_prep_s": prep_s, "cache_struct_s": cache_struct, "ref_struct_s": ref_struct,
                       "cache_gib": cache_gib, "ref_gib": ref_gib, "direct_gib": dir_gib,
                       "ref_over_cache": ref_s / cache_s, "direct_over_cache": dir_s / cache_s if math.isfinite(dir_s) else np.nan})
        del model, batches, preps, prep0
        torch.cuda.empty_cache()

pd.set_option("display.float_format", lambda v: f"{v:.4g}")
eq = pd.DataFrame(rows_eq)
print("EQUIVALENCE"); print(eq.to_string(index=False))
dft = pd.DataFrame(rows_t)
print("EXECUTION COST BY CLASS"); print(dft.to_string(index=False))
dsw = pd.DataFrame(sweep)
print("CHUNK SIZE SWEEP (full class)"); print(dsw.to_string(index=False))

est = []
for kind in ("W",):
    d = dft[dft["arm"] == kind].set_index("class")
    hours = lambda col: sum(float(col[c]) * counts[c] for c in CLASSES) / 3600
    ok = dsw[(dsw["arm"] == kind) & (dsw["status"] == "ok")].sort_values("direct_exec_s")
    dex = d["direct_exec_s"].astype(float).copy()
    best_chunk = d.loc["full", "direct_chunk"]
    if len(ok):
        best_chunk, dex["full"] = int(ok.iloc[0]["chunk_rows"]), float(ok.iloc[0]["direct_exec_s"])
    direct_ok = bool(np.isfinite(dex.values).all())
    r = {"arm": kind, "cache_exec_h": hours(d["cache_exec_s"]), "ref_exec_h": hours(d["ref_exec_s"]),
         "direct_exec_h": hours(dex) if direct_ok else np.nan, "direct_best_chunk": best_chunk,
         "cache_struct_h": hours(d["cache_struct_s"]), "ref_struct_h": hours(d["ref_struct_s"]),
         "direct_prep_h": hours(d["direct_prep_s"])}
    assert np.isfinite([r["cache_exec_h"], r["ref_exec_h"], r["cache_struct_h"], r["ref_struct_h"], r["direct_prep_h"]]).all()
    r["ref_over_cache"] = r["ref_exec_h"] / r["cache_exec_h"]
    r["direct_over_cache"] = r["direct_exec_h"] / r["cache_exec_h"] if direct_ok else np.nan
    r["cache_total_h"] = r["cache_exec_h"] + r["cache_struct_h"]
    r["ref_total_h"] = r["ref_exec_h"] + r["ref_struct_h"]
    r["direct_total_h"] = r["direct_exec_h"] + r["direct_prep_h"] + r["ref_struct_h"] if direct_ok else np.nan
    r["direct_projection"] = "ok" if direct_ok else "unavailable: a class has no chunk that ran all six microbatches"
    est.append(r)
hours_tbl = pd.DataFrame(est)
print(f"HOURS PER EPOCH (class counts: {', '.join(f'{c} {n:,}' for c, n in counts.items())}; forward and backward only)")
print(hours_tbl.to_string(index=False))

COST = os.path.join(RUN_DIR, "v3_cost_study.pkl.lz4")
save_to_pickle({"written": datetime.datetime.now().isoformat(timespec="seconds"), "sources_sha256": SOURCES_SHA256, "environment": ENVIRONMENT,
                "settings": {"PER_CLASS": PER_CLASS, "CHUNK_ROWS": CHUNK_ROWS, "CHUNK_SWEEP": CHUNK_SWEEP, "ATOL": ATOL, "RTOL": RTOL,
                             "ref_walk": REF_WALK, "picked_microbatches": picked, "class_counts": counts},
                "tables": {"equivalence": eq, "execution_by_class": dft, "chunk_sweep": dsw, "hours_per_epoch": hours_tbl}},
               COST, lz4_compression=True)                                                        # atomic: mkstemp in the same folder, then os.replace
print("saved"); print(pd.DataFrame([{"path": COST, "kib": os.path.getsize(COST) / 1024, "sha16": sha256_file(COST)[:16]}]).to_string(index=False))
pd.reset_option("display.float_format")

picked microbatches
 class  epoch_count                                       picked
single        61105 [16179, 24651, 52250, 71511, 119850, 122502]
 small        29405 [18384, 84989, 88691, 91908, 101659, 102626]
medium         4794  [8331, 61610, 63091, 72135, 119756, 123317]
  full        31463    [97, 51450, 60107, 63126, 119277, 123343]
EQUIVALENCE
arm  class           path  chunk_rows status  rel_loss_diff               worst_param  worst_err_over_tol  subsets  usable
  W single reference walk         NaN     ok      1.781e-07 encoder.blocks.0.3.weight            0.001825     2002    1004
  W single         direct         256     ok      1.781e-07            query.mask_vec            0.002553     2002    1004
  W single         direct        1024     ok      8.904e-08            query.mask_vec             0.00217     2002    1004
  W  small reference walk         NaN     ok              0          readout.1.weight            0.002138     5869    2614
  W  small         direct   

## Cell 9. Tables for Section 8

Reads the saved report (cell 7: endpoints and tradeoffs), the three saved curves (cell 6: epoch losses, probe and hours) and the cost study (cell 8), and writes three LaTeX tables to `pilot\tables`: `tab_pilot_baseline`, `tab_pilot_endpoints` and `tab_pilot_bins_sampled`. Each curve's contract is checked against the one the report used. Nothing in Section 8 is typed by hand. The cell ends with the unrounded hours per epoch and ratios from the cost study, the numbers Section 8.6 quotes.

Printed labels, in order:

- `report`: endpoint rows, tradeoff rows and arm epochs read.
- One line per table written, with its character count.
- `tab_pilot_endpoints` in full.
- The cost study column: `cache_exec_h`, `direct_exec_h`, `direct_prep_h`, `ref_exec_h`, `cache_struct_h`, `direct_over_cache`, `with_prep_over_cache` (direct execution plus direct preparation over cache execution), `ref_over_cache` and the cost study file's `sha16`.

In [8]:
# Estimated run time: under 5 seconds. Pilot tables for the paper, generated from the final pilot results, nothing typed by hand.
# Reads the saved report (cell 7), the three saved curves (cell 6) and the cost study (cell 8), writes three LaTeX tables to
# pilot\tables, and prints the unrounded hours per epoch and ratios for Section 8.6. Runs after cell 8, same kernel. Reads files
# only, and keeps its work inside one function, so it changes no name the earlier cells use (GROUPS, endpoints, ...).
PT_REPORT = os.path.join(RUN_DIR, "v3_pilot_report.pkl.lz4")
PT_COST = os.path.join(RUN_DIR, "v3_cost_study.pkl.lz4")
PT_TABLE_DIR = os.path.join(PILOT_DIR, "tables")

def build_pilot_tables():
    arms = ["W0_registered", "W0", "W"]
    comparison = "W minus W0"
    groups = [("pooled", None), ("0 to 1", "0 to 1"), ("1 to 3", "1 to 3"), ("3 to 10", "3 to 10"),
              ("common", "seen_common"), ("rare", "seen_rare"), ("unseen", "unseen")]
    bins = [f"bin {i}" for i in range(10)]

    # ----- the report and the curves it was computed from -----
    report = load_from_pickle(PT_REPORT)
    assert report is not None, f"report not found at {PT_REPORT}"
    end = report["tables"]["endpoints"].copy()
    trd = report["tables"]["tradeoffs"].copy()
    rows = []
    for arm in arms:
        path = os.path.join(RUN_DIR, f"v3_{arm}_curve.pkl.lz4")
        cv = load_from_pickle(path)
        assert cv is not None, f"curve not found at {path}"
        assert cv["contract"] == report["contracts"][arm], f"{arm}: curve contract differs from the one the report used"
        hours = 0.0
        for ep in (1, 2):
            info = cv["epochs"][ep]
            hours += info["train_h"]
            rows.append({"arm": arm, "epoch": ep, **{k: info["row"][k] for k in ("heldout", "0 to 1", "1 to 3", "3 to 10",
                         "seen_common", "seen_rare", "unseen", "d_pred_shuffle")}, "hours_cum": hours})
    epo = pd.DataFrame(rows)
    for f in (end, trd):
        f["epoch"] = pd.to_numeric(f["epoch"])
    print(f"report: {len(end)} endpoint rows, {len(trd)} tradeoff rows; curves: {len(epo)} arm epochs")

    def one(df, **match):
        sel = pd.Series(True, index=df.index)
        for k, v in match.items():
            sel &= df[k] == v
        hit = df[sel]
        assert len(hit) == 1, f"expected one row for {match}, found {len(hit)}"
        return hit.iloc[0]

    def num(v, d=4):
        return f"{v:.{d}f}"

    def signed(v, d=4):
        s = f"{v:+.{d}f}"
        return "+" + s[1:] if float(s) == 0 else s          # no negative zero

    def interval(lo, hi):
        return f"$[{signed(lo)}, {signed(hi)}]$"

    def tex(caption, label, spec, header, body, colsep=None, spanner=()):
        out = [r"\begin{table}[t]", f"\\caption{{{caption}}}", f"\\label{{{label}}}", r"\small"]
        if colsep:
            out.append(f"\\setlength{{\\tabcolsep}}{{{colsep}}}")
        out += [f"\\begin{{tabular}}{{{spec}}}", r"\toprule", *spanner, " & ".join(header) + r" \\", r"\midrule"]
        out += [" & ".join(r) + r" \\" for r in body] + [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
        return "\n".join(out)

    # ----- the baseline under the two update rules -----
    base = []
    for rule, arm in (("registered", "W0_registered"), ("accumulated", "W0")):
        for ep in (1, 2):
            r = one(epo, arm=arm, epoch=ep)
            p = r["d_pred_shuffle"]
            base.append([rule, str(ep), num(r["heldout"]), num(r["0 to 1"]), num(r["1 to 3"], 3), num(r["3 to 10"], 3),
                         num(r["seen_common"]), num(r["seen_rare"]), num(r["unseen"]), num(p, 4 if p < 0.01 else 3),
                         num(r["hours_cum"], 2)])
    t_base = tex("The baseline under the registered and the accumulated update rule on the filtered data. Held out Huber "
                 "loss, pooled, by band and by concept group. Probe is the median absolute change in prediction under the "
                 "permutation probe of Section~\\ref{sec:pilot:baseline}. Hours are cumulative training time. References are "
                 "zero at $0.2346$ and the concept median at $0.2242$.",
                 "tab:baseline", "llrrrrrrrrr",
                 ["Rule", "Epoch", "Pooled", "0 to 1", "1 to 3", "3 to 10", "Common", "Rare", "Unseen", "Probe", "Hours"],
                 base, colsep="4pt")

    # ----- epoch 2 paired differences by band and concept group -----
    body = []
    for name, group in groups:
        r = one(end, comparison=comparison, epoch=2) if group is None else one(trd, comparison=comparison, epoch=2, group=group)
        if group is None:
            n = sum(int(one(trd, comparison=comparison, epoch=2, group=b)["targets"]) for b in bins)
        else:
            n = int(one(trd, comparison=comparison, epoch=2, group=group)["targets"])
        body.append([name, f"{n:,}", f"${signed(r['d_mean'])}$", interval(r["ci_lo"], r["ci_hi"])])
    t_end = tex("Paired differences at epoch 2, pooled, by band and by concept group. Targets is the number of held out "
                "targets in the group. Paired difference is the subset supervised model's mean held out Huber loss minus "
                "the baseline's on the same targets, negative favors the subset supervised model. Intervals are record "
                "clustered $95\\%$ bootstrap percentile intervals, nominal for subgroups.",
                "tab:endpoints", "lrrl",
                ["Group", "Targets", "Paired difference", "Interval"], body)

    # ----- paired differences by bin at both epochs -----
    body = []
    for i, b in enumerate(bins):
        r1, r2 = one(trd, comparison=comparison, epoch=1, group=b), one(trd, comparison=comparison, epoch=2, group=b)
        assert int(r1["targets"]) == int(r2["targets"]), f"{b}: target counts differ between epochs"
        body.append([str(i), f"{int(r1['targets']):,}", f"${signed(r1['d_mean'])}$", interval(r1["ci_lo"], r1["ci_hi"]),
                     f"${signed(r2['d_mean'])}$", interval(r2["ci_lo"], r2["ci_hi"])])
    spanner = [r" & & \multicolumn{2}{c}{Epoch 1} & \multicolumn{2}{c}{Epoch 2} \\", r"\cmidrule(lr){3-4} \cmidrule(lr){5-6}"]
    t_bins = tex("Paired differences by bin, subset supervised model minus baseline. Bins are the ten subgroups of held out "
                 "targets by the magnitude of the signed log target $|g(v)|$, with $g(v) = \\mathrm{sign}(v) \\log(1 + |v|)$, "
                 "cut at the training deciles, bin 0 the smallest. Targets is the number of held out targets in the bin. "
                 "Paired difference is the subset supervised model's mean held out Huber loss minus the baseline's on the "
                 "same targets, negative favors the subset supervised model. Intervals are record clustered $95\\%$ bootstrap "
                 "percentile intervals, nominal for subgroups.",
                 "tab:pilot_bins_sampled", "rrrlrl", ["Bin", "Targets", "Paired difference", "Interval", "Paired difference", "Interval"],
                 body, spanner=spanner)

    return {"tab_pilot_baseline": t_base, "tab_pilot_endpoints": t_end, "tab_pilot_bins_sampled": t_bins}

pilot_tables = build_pilot_tables()
os.makedirs(PT_TABLE_DIR, exist_ok=True)
for pt_name, pt_body in pilot_tables.items():
    with open(os.path.join(PT_TABLE_DIR, f"{pt_name}.tex"), "w", encoding="utf-8") as pt_f:
        pt_f.write(pt_body)
    print(f"{pt_name}: {len(pt_body):,} characters written")
print("\n" + pilot_tables["tab_pilot_endpoints"])

# ----- unrounded hours per epoch and ratios from the cost study, for Section 8.6 -----
cs = load_from_pickle(PT_COST)
h = cs["tables"]["hours_per_epoch"].iloc[0]
with_prep = (h["direct_exec_h"] + h["direct_prep_h"]) / h["cache_exec_h"]
pd.DataFrame([{"cache_exec_h": h["cache_exec_h"], "direct_exec_h": h["direct_exec_h"], "direct_prep_h": h["direct_prep_h"],
               "ref_exec_h": h["ref_exec_h"], "cache_struct_h": h["cache_struct_h"],
               "direct_over_cache": h["direct_over_cache"], "with_prep_over_cache": with_prep,
               "ref_over_cache": h["ref_over_cache"], "sha16": sha256_file(PT_COST)[:16]}]).T

report: 4 endpoint rows, 32 tradeoff rows; curves: 6 arm epochs
tab_pilot_baseline: 1,007 characters written
tab_pilot_endpoints: 942 characters written
tab_pilot_bins_sampled: 1,738 characters written

\begin{table}[t]
\caption{Paired differences at epoch 2, pooled, by band and by concept group. Targets is the number of held out targets in the group. Paired difference is the subset supervised model's mean held out Huber loss minus the baseline's on the same targets, negative favors the subset supervised model. Intervals are record clustered $95\%$ bootstrap percentile intervals, nominal for subgroups.}
\label{tab:endpoints}
\small
\begin{tabular}{lrrl}
\toprule
Group & Targets & Paired difference & Interval \\
\midrule
pooled & 136,446 & $-0.0079$ & $[-0.0091, -0.0069]$ \\
0 to 1 & 119,861 & $-0.0015$ & $[-0.0020, -0.0010]$ \\
1 to 3 & 14,846 & $-0.0471$ & $[-0.0533, -0.0408]$ \\
3 to 10 & 1,739 & $-0.1169$ & $[-0.1587, -0.0731]$ \\
common & 111,029 & $-0.0084$ & $[-0.0096, -0.0073]$ 

,0
cache_exec_h,1.83886
direct_exec_h,2.161411
direct_prep_h,1.436548
ref_exec_h,2.609192
cache_struct_h,3.220137
direct_over_cache,1.175408
with_prep_over_cache,1.956625
ref_over_cache,1.418918
sha16,dab57d5e88a2f098


## Cell 10. Parameter count

Counts the trainable parameters of the pilot model from the saved final checkpoint of the subset supervised model. The baseline has the same architecture, so the count is the same for both. Records the count for the README (Q10.8).

In [2]:
# Estimated run time: under 10 seconds. Runs after Cell 2, any kernel that has run Cell 2.
import torch

CKPT = os.path.join(RUN_DIR, "v3_W_last.pt")
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)

print("top level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
state = None
for k in ("model", "model_state", "model_state_dict", "state_dict"):
    if isinstance(ckpt, dict) and k in ckpt:
        state = ckpt[k]
        print("using key:", k)
        break
if state is None and isinstance(ckpt, dict) and all(torch.is_tensor(v) for v in ckpt.values()):
    state = ckpt
    print("checkpoint is a bare state dict")
assert state is not None, "no model state found under the expected keys; see the printed top level keys"

rows = [{"tensor": k, "numel": v.numel()} for k, v in state.items() if torch.is_tensor(v) and v.is_floating_point()]
counts = pd.DataFrame(rows)
print(f"floating point tensors: {len(counts):,}")
print(f"parameters: {int(counts['numel'].sum()):,}")
counts.sort_values("numel", ascending=False).head(10)

top level keys: ['model', 'opt', 'st', 'curve', 'contract', 'rng_cpu', 'rng_cuda']
using key: model
floating point tensors: 35
parameters: 1,647,627


,tensor,numel
5,encoder.blocks.0.1.weight,262144
13,encoder.blocks.1.3.weight,262144
7,encoder.blocks.0.3.weight,262144
11,encoder.blocks.1.1.weight,262144
29,readout.1.weight,132352
22,query.adapter.weight,98304
2,encoder.adapter.weight,98304
19,encoder.v.weight,65536
25,query.q.weight,65536
17,encoder.k.weight,65536
